# AD-EEG — frozen CBraMod (TDBRAIN·Mumtaz·PD-EEG와 같은 프로토콜, 별도 과제)

**시작 전:** 메뉴 `런타임 → 런타임 유형 변경 → GPU` 선택 후, 위에서부터 한 셀씩 실행하세요.

- **데이터:** OpenNeuro ds004504 v1.0.9 (그리스 AHEPA 병원, AD 36 / FTD 23 / CN 29, 눈 감은 안정 상태 평균 약 13분, 19채널 10-20, A1·A2 기준, 500 Hz, 기록 장비 필터 0.4–50 Hz)
- **임상정보:** Age, Gender, MMSE. **CN은 MMSE가 전원 30점**이라 AD vs CN에서는 MMSE 하나로 100% 구분됨 (누출 성격)

| 과제 (`--task`) | 내용 | 역할 |
|---|---|---|
| **ad_cn** | AD(1) vs CN(0) | 주 과제 (PD-EEG의 PD vs HC에 해당) |
| **ad_ftd** | AD(1) vs FTD(0) | MMSE가 겹쳐 정답을 그대로 알려주지 않는 과제 (PD-EEG 과제 C에 해당) |

| 실험 | AD vs CN | AD vs FTD | 비고 |
|---|---|---|---|
| 1. EEG-only (+ seed 10개) | ✅ | ✅ | 모든 비교의 기준선 |
| 2. 잡음 점검 | ✅ | ✅ | EC만 있음 → 잡음 점검 중심 |
| 추가 B. EEG–임상정보 관련성 | ✅ | ✅ | EEG로 MMSE·나이 예측 |
| 추가 A. EEG 근거 보존 결합 | ✅ | ✅ | AD vs CN은 단순 결합 행이 "MMSE 누출 대조·정보 교체 재현" 역할 |
| 3. 임상정보 조합 · 4. 정보 교체 | ❌ (생략) | ✅ | AD vs CN은 MMSE가 완벽한 누출, 나이·성별은 matched에서 정보 없음 → 결과가 정해져 있음 |
| **AD 고유 1. 기록 길이 학습 곡선** | ✅ | ✅ | 앞 1·2·4·8분 / 전체 → 데이터셋 간 기록 길이 차이 보정 |
| **AD 고유 2. 서파화 지표 비교** | ✅ | ✅ | 고전 밴드 파워·서파화 지표 vs CBraMod, CBraMod 점수가 서파화를 따라가는지 |
| **AD 고유 3. 제공자 전처리본(ICA·ASR) 비교** | ✅ | ✅ | 잡음을 전문적으로 제거해도 성능이 유지되는지 |

**같은 것:** 19채널(Mumtaz와 같은 이름·순서) · 0.3–75 Hz · 50 Hz notch · 200 Hz · 5 s 비중첩 · μV/100 · frozen CBraMod(원본 repo + 원본 가중치) · StandardScaler + L2 LR(C=0.01) · seed 42 · 결합 모델 C=1.0
**다른 것:** EC만 있음 · 기록이 길어 기록당 약 150구간 · 특징은 88명 전원 한 번만 추출해 두 과제가 공유

| 후속 실험 | 내용 |
|---|---|
| 전처리본으로 실험 3·4·A·B 반복 | AD vs CN, 잡음 제거 후에도 결론이 같은지 |
| CBraMod + 고전 지표 결합 | PAF, theta/alpha, 서파화 지표와 CBraMod 점수 결합 (`ad_more.py`) |

> **데이터는 저장소에 포함되지 않습니다.** OpenNeuro 공개 데이터를 실행 중에 받습니다.

## 0. 설치

In [ ]:
!pip install -q mne openneuro-py einops huggingface_hub
import torch; print('GPU:', torch.cuda.is_available())

## 1. Google Drive 연결 + 경로 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, glob, json
WORK  = '/content/drive/MyDrive/ad_eeg_project'      # 결과·데이터 백업 위치
LOCAL = '/content/ad_eeg'
DRIVE_DATA = f'{WORK}/ds004504'
EEG_ROOT = f'{LOCAL}/ds004504'
PARTS = f'{EEG_ROOT}/participants.tsv'
FEAT  = f'{WORK}/features'                           # 원본에서 뽑은 특징 (두 과제 공유)
FEAT_DERIV = f'{WORK}/features_deriv'                # 제공자 전처리본에서 뽑은 특징 (AD 고유 3)
SPEC  = f'{WORK}/spectral.csv'                       # 고전 지표 (AD 고유 2)
TASKS = ['ad_cn', 'ad_ftd']
OUTS  = {t: f'{WORK}/out_{t}' for t in TASKS + ['all']}
OUTS_D = {t: f'{WORK}/out_{t}_deriv' for t in TASKS + ['all']}
for p in [FEAT, FEAT_DERIV] + list(OUTS.values()) + list(OUTS_D.values()): os.makedirs(p, exist_ok=True)

def run(step, task, extra='', deriv=False):   # ad_cbramod.py (실험 1·2·3·4, 추출)
    o, f = (OUTS_D[task], FEAT_DERIV) if deriv else (OUTS[task], FEAT)
    get_ipython().system(f'python /content/ad_cbramod.py {step} --participants "{PARTS}" --out "{o}" --task {task} --feat-dir "{f}" {extra}')
def runx(step, task, extra=''):               # ad_extra.py (추가 실험 B·A)
    get_ipython().system(f'python /content/ad_extra.py {step} --participants "{PARTS}" --out "{OUTS[task]}" --task {task} --feat-dir "{FEAT}" {extra}')
def runs(step, task, extra=''):               # ad_special.py (AD 고유 실험 1·2)
    get_ipython().system(f'python /content/ad_special.py {step} --participants "{PARTS}" --out "{OUTS[task]}" --task {task} --feat-dir "{FEAT}" --spec-csv "{SPEC}" {extra}')

## 1-1. CBraMod 원본 코드 + 사전학습 가중치
TDBRAIN·Mumtaz·PD-EEG와 같은 조합. 다른 프로젝트 폴더에 이미 받아 둔 것이 있으면 그대로 씁니다.

In [ ]:
CBRAMOD_REPO, WEIGHTS = None, None
for base in [WORK, '/content/drive/MyDrive/pd_eeg_project', '/content/drive/MyDrive/mumtaz_hcmdd_project', '/content/drive/MyDrive/tdbrain_hcmdd_project']:
    if os.path.exists(f'{base}/CBraMod/models/cbramod.py') and os.path.exists(f'{base}/pretrained_weights.pth'):
        CBRAMOD_REPO, WEIGHTS = f'{base}/CBraMod', f'{base}/pretrained_weights.pth'; break
if CBRAMOD_REPO is None:
    from huggingface_hub import hf_hub_download
    CBRAMOD_REPO, WEIGHTS = f'{WORK}/CBraMod', f'{WORK}/pretrained_weights.pth'
    !git clone -q https://github.com/wjq-learning/CBraMod "$CBRAMOD_REPO"
    shutil.copy(hf_hub_download('weighting666/CBraMod', 'pretrained_weights.pth'), WEIGHTS)
print('repo:', CBRAMOD_REPO, '| weights MB:', round(os.path.getsize(WEIGHTS) / 1e6, 1))   # 약 19.8 MB

## 2. EEG 데이터 준비 (원본만)
제공자 전처리본(`derivatives/`)은 AD 고유 3에서 따로 받습니다. Drive에 받아 둔 데이터가 있으면 로컬로 복사만 합니다.

In [ ]:
def n_set(p): return len(glob.glob(f'{p}/sub-*/**/*_eeg.set', recursive=True))

if os.path.isdir(DRIVE_DATA) and n_set(DRIVE_DATA) >= 88:
    print('Drive에 데이터 있음 -> 로컬로 복사')
    shutil.copytree(DRIVE_DATA, EEG_ROOT, dirs_exist_ok=True, ignore=shutil.ignore_patterns('derivatives'))
else:
    os.makedirs(EEG_ROOT, exist_ok=True)
    try:
        import openneuro
        openneuro.download(dataset='ds004504', tag='1.0.9', target_dir=EEG_ROOT,
                           include=['participants.tsv', 'participants.json', 'dataset_description.json', 'sub-*'])
    except Exception as e:
        print('openneuro-py 실패 -> AWS S3 공개 버킷으로 받기:', e)
        !pip install -q awscli
        !aws s3 sync --no-sign-request s3://openneuro.org/ds004504 "$EEG_ROOT" --exclude "*" --include "participants.*" --include "dataset_description.json" --include "sub-*"
    print('Drive에 백업 중...')
    shutil.copytree(EEG_ROOT, DRIVE_DATA, dirs_exist_ok=True)

print('원본 .set 파일 수:', n_set(EEG_ROOT), '(88개면 정상)')
print('participants.tsv:', os.path.exists(PARTS))

## 3. 스크립트 파일 만들기 (4개)
- `ad_cbramod.py`: 실험 1~4 (inspect / select / extract / cv / seeds / noise / clinical / swap)
- `ad_extra.py`: 추가 실험 B·A (relate / preserve)
- `ad_special.py`: AD 고유 실험 (length / spectral / compare)
- `ad_more.py`: 후속 실험 (CBraMod + 고전 지표 결합)

In [ ]:
%%writefile /content/ad_cbramod.py
"""
AD-EEG (OpenNeuro ds004504) — frozen CBraMod + L2 logistic regression
(TDBRAIN·Mumtaz·PD-EEG와 같은 프로토콜을 별도 과제로 적용)

과제 (--task)
  ad_cn  : AD(1) vs CN(0)      — 주 과제
  ad_ftd : AD(1) vs FTD(0)     — 두 치매 구분 (MMSE가 정답을 그대로 알려주지 않는 과제)
  ftd_cn : FTD(1) vs CN(0)
  all    : 특징 추출용 (88명 전원)

단계
  inspect  : participants.tsv 열·그룹별 값, 첫 EEG 파일의 채널·샘플링·진폭 확인, 임상 변수 분류 제안
  select   : 피험자 선정 (full, matched)
  extract  : 안정 상태(눈 감음) EEG 전처리 -> frozen CBraMod 특징(3,800차원), 기록 단위 캐시
  cv / seeds / noise / clinical / swap : 실험 1~4 (PD-EEG와 동일)

TDBRAIN·Mumtaz·PD-EEG와 다른 점
  - 안정 상태 기록이 눈 감은 상태(EC) 하나뿐
  - 19채널 10-20 (Mumtaz와 같은 이름·순서), 원본 500 Hz -> 200 Hz
  - 기록이 길다 (평균 약 13분 -> 기록당 약 150구간)
"""
import argparse, glob, itertools, json, os, re, sys, time
import numpy as np
import pandas as pd

# ----------------------------------------------------------------------------- 설정 (TDBRAIN·Mumtaz와 동일)
SEED = 42
FS_TARGET = 200
SEG_SEC = 5
L_FREQ, H_FREQ, NOTCH = 0.3, 75.0, 50.0
AMP_DIV = 100.0              # μV / 100
FLAG_UV = 100.0              # 품질 검토용 표시 기준(제외에는 사용 안 함)
C_REG = 0.01                 # EEG 분류기
C_FUSION = 1.0               # 임상정보 결합 분류기 (TDBRAIN 실험 3과 동일)
N_SWAP = 20                  # 실험 4 기증자 추출 횟수
MAX_COMBO_VARS = 10          # 실험 3 조합에 쓸 최대 변수 수 (2^10-1 = 1,023개 조합)
CONDITIONS = ["EC"]          # 안정 상태 = 눈 감음
COG_CUT = 26                 # task=cog: PD 환자 중 MoCA < 26 을 인지 저하(1)로
N0, N1 = "CN", "AD"          # 라벨 0/1 이름 (과제에 따라 바뀜)
TASKS = {"ad_cn": ("C", "A", "CN", "AD"), "ftd_cn": ("C", "F", "CN", "FTD"), "ad_ftd": ("F", "A", "FTD", "AD")}

# Mumtaz 19채널 순서 (10-10 이름: T3->T7, T4->T8, T5->P7, T6->P8)
TARGET_19 = ["Fp1", "Fp2", "F7", "F3", "Fz", "F4", "F8", "T7", "C3", "Cz",
             "C4", "T8", "P7", "P3", "Pz", "P4", "P8", "O1", "O2"]
ALIAS = {"t3": "t7", "t4": "t8", "t5": "p7", "t6": "p8"}
SYMPTOM_KEYS = ["updrs", "hy", "hoehn", "yahr", "moca", "mmse", "bdi", "gds", "hads", "hamd", "depress",
                "anx", "ctt", "fog", "tremor", "ledd", "levodopa", "duration", "medic", "pdq", "nmss", "tmt", "tug"]


# ----------------------------------------------------------------------------- 참가자 표
def _find_col(d, pattern):
    for c in d.columns:
        if re.search(pattern, c, re.IGNORECASE):
            return c
    return None


def detect_label_col(d):
    for c in d.columns:
        if c == "participant_id":
            continue
        v = d[c].dropna().astype(str).str.strip().str.upper()
        if 1 < v.nunique() <= 6 and v.str.contains(r"PD|PARK").any() and v.str.contains(r"HC|CONTROL|HEALTHY|CTRL").any():
            return c
    return None


def load_participants(path, label_col=None, task="ad_cn"):
    """Group 열: A = AD, F = FTD, C = CN. 과제에 맞는 두 그룹만 남기고 label 0/1 부여"""
    global N0, N1
    d = pd.read_csv(path, sep="\t", na_values=["n/a", "N/A", "NA", "nan", "NaN", ""])
    d["subject"] = d["participant_id"].astype(str)
    gcol = label_col or _find_col(d, r"^group$")
    if gcol is None:
        raise RuntimeError(f"Group 열을 찾지 못했습니다. 열 목록: {list(d.columns)} -> --label-col 로 지정")
    g = d[gcol].astype(str).str.strip().str.upper()
    if task == "all":
        d = d.copy(); d["label"] = (g != "C").astype(int); N0, N1 = "CN", "치매"
    else:
        neg, pos, N0, N1 = TASKS[task]
        keep = g.isin([neg, pos])
        d = d[keep].copy(); d["label"] = (g[keep] == pos).astype(int)
    d.attrs["task"] = task
    d.attrs["label_col"] = gcol
    d.attrs["age_col"] = _find_col(d, r"^age")
    d.attrs["sex_col"] = _find_col(d, r"^(sex|gender)")
    return d.reset_index(drop=True)


def select_subjects(d, cohort="full", caliper=3.0):
    if cohort == "full":
        return d.copy()
    if cohort != "matched":
        raise ValueError(cohort)
    age, sex = d.attrs.get("age_col"), d.attrs.get("sex_col")
    if age is None or sex is None:
        raise RuntimeError("나이·성별 열이 없어 matched 코호트를 만들 수 없습니다. --cohort full 사용")
    dd = d[d[age].notna() & d[sex].notna()]
    hc = dd[dd.label == 0].sample(frac=1, random_state=SEED)
    pool = dd[dd.label == 1].copy()
    keep = []
    for _, r in hc.iterrows():                                # HC 1명당 같은 성별·나이 최근접 PD 1명
        cand = pool[pool[sex] == r[sex]]
        if cand.empty:
            continue
        diff = (cand[age] - r[age]).abs()
        if diff.min() > caliper:
            continue
        j = diff.idxmin()
        keep += [r.subject, pool.loc[j, "subject"]]
        pool = pool.drop(j)
    out = d[d.subject.isin(keep)].reset_index(drop=True)
    out.attrs = d.attrs
    return out


def describe(d):
    age, sex = d.attrs.get("age_col"), d.attrs.get("sex_col")
    rows = []
    for lab, g in d.groupby("label"):
        r = {"group": N1 if lab else N0, "n": len(g)}
        if age:
            r.update(age_mean=round(g[age].mean(), 1), age_sd=round(g[age].std(), 1))
        if sex:
            r.update({f"{sex}={k}": int(v) for k, v in g[sex].value_counts().items()})
        rows.append(r)
    return pd.DataFrame(rows)


# ----------------------------------------------------------------------------- EEG 읽기·전처리
def find_file(eeg_root, sid):
    hits = []
    for ext in ("set", "edf", "bdf", "vhdr", "fif"):
        hits += glob.glob(os.path.join(eeg_root, "**", f"{sid}_*task-*_eeg.{ext}"), recursive=True)
    hits.sort(key=lambda h: ("derivatives" in h.lower(), h))
    return hits[0] if hits else None


def _norm(ch):
    c = ch.lower().replace("eeg", "").strip(" -_")
    c = re.split(r"[-_ ]", c)[0]
    return ALIAS.get(c, c)


def load_raw(path):
    """19채널을 이름으로 골라 Mumtaz 순서로 정렬. 없거나 신호가 0인 채널은 보간. (raw, 보간한 채널 목록) 반환"""
    import mne
    raw = mne.io.read_raw_eeglab(path, preload=True, verbose="ERROR") if path.lower().endswith(".set") \
        else mne.io.read_raw(path, preload=True, verbose="ERROR")
    mont = mne.channels.make_standard_montage("standard_1005")
    canon = {n.lower(): n for n in mont.ch_names}
    rename, used = {}, set()
    for c in raw.ch_names:
        k = _norm(c)
        if k in canon and canon[k] not in used:
            rename[c] = canon[k]; used.add(canon[k])
    if "Cz" not in used:                                      # HydroCel 기준전극(VREF)은 Cz 위치
        for c in raw.ch_names:
            if c.upper() in ("VREF", "REF", "E65", "CZ_REF") and c not in rename:
                rename[c] = "Cz"; used.add("Cz"); break
    raw = raw.pick(list(rename)).rename_channels(rename)
    raw.set_channel_types({c: "eeg" for c in raw.ch_names}, verbose="ERROR")
    missing = [t for t in TARGET_19 if t not in raw.ch_names]
    if missing:                                               # 없는 채널은 0으로 추가한 뒤 보간
        info = mne.create_info(missing, raw.info["sfreq"], "eeg")
        raw.add_channels([mne.io.RawArray(np.zeros((len(missing), raw.n_times)), info, verbose="ERROR")],
                         force_update_info=True)
    raw.set_montage(mont, match_case=False, on_missing="ignore", verbose="ERROR")
    x = raw.get_data(picks=TARGET_19)
    flat = [t for t, s in zip(TARGET_19, x.std(1)) if s < 1e-9 and t not in missing]
    bads = missing + flat
    if bads:
        if len(raw.ch_names) - len(bads) < 20:
            raise RuntimeError(f"보간에 쓸 채널이 부족합니다: {raw.ch_names}")
        raw.info["bads"] = bads
        raw.interpolate_bads(reset_bads=True, origin="auto", verbose="ERROR")
    raw.pick(TARGET_19)
    return raw, bads


def preprocess(path):
    """reference 유지, 0.3–75 Hz, 50 Hz notch, 200 Hz, 5 s 비중첩, μV/100 -> [N,19,5,200]"""
    raw, interp = load_raw(path)
    fs_orig = raw.info["sfreq"]
    raw.filter(L_FREQ, H_FREQ, verbose="ERROR")
    raw.notch_filter(NOTCH, verbose="ERROR")
    raw.resample(FS_TARGET, verbose="ERROR")
    x = raw.get_data() * 1e6                                  # V -> μV
    n = x.shape[1] // (FS_TARGET * SEG_SEC)
    x = x[:, : n * FS_TARGET * SEG_SEC]
    segs = x.reshape(19, n, SEG_SEC, FS_TARGET).transpose(1, 0, 2, 3)
    segmax = np.abs(segs).reshape(n, -1).max(1)                # 구간별 최대 진폭(μV): 잡음 기준을 나중에 바꿀 수 있게 저장
    flag = segmax > FLAG_UV
    return (segs / AMP_DIV).astype(np.float32), flag, fs_orig, x.shape[1] / FS_TARGET, interp, segmax.astype(np.float32)


class _BraindecodeWrap:
    def __init__(self, m): self.m = m
    def parameters(self): return self.m.parameters()
    def __call__(self, x): return self.m(x.reshape(x.shape[0], x.shape[1], -1))


def build_cbramod(repo, weights):
    import torch
    if repo is None:
        from braindecode.models import CBraMod
        m = CBraMod.from_pretrained("braindecode/cbramod-pretrained", return_encoder_output=True).eval()
        m = m.to("cuda" if torch.cuda.is_available() else "cpu")
        for p in m.parameters(): p.requires_grad_(False)
        return _BraindecodeWrap(m)
    sys.path.insert(0, repo)
    from models.cbramod import CBraMod
    m = CBraMod(in_dim=200, out_dim=200, d_model=200, dim_feedforward=800, seq_len=30, n_layer=12, nhead=8)
    m.load_state_dict(torch.load(weights, map_location="cpu"))
    m.proj_out = torch.nn.Identity()
    m = m.to("cuda" if torch.cuda.is_available() else "cpu").eval()
    for p in m.parameters(): p.requires_grad_(False)
    return m


def extract_features(model, segs, bs=64):
    import torch
    dev = next(model.parameters()).device
    out = []
    with torch.no_grad():
        for i in range(0, len(segs), bs):
            z = model(torch.from_numpy(segs[i:i + bs]).to(dev))
            out.append(z.mean(dim=2).reshape(z.shape[0], -1).cpu().numpy())
    return np.concatenate(out).astype(np.float32)


# ----------------------------------------------------------------------------- inspect
def run_inspect(a):
    d = load_participants(a.participants, a.label_col, a.task)
    os.makedirs(a.out, exist_ok=True)
    print("라벨 열:", d.attrs["label_col"], "| 나이 열:", d.attrs["age_col"], "| 성별 열:", d.attrs["sex_col"])
    print(describe(d).to_string(index=False), "\n")
    rows = []
    skip = {"participant_id", "subject", "label", d.attrs["label_col"]}
    for c in d.columns:
        if c in skip:
            continue
        hc, pdg = d[d.label == 0][c], d[d.label == 1][c]
        num = pd.to_numeric(d[c], errors="coerce")
        r = {"column": c, f"{N0} 값 있음 %": round(hc.notna().mean() * 100), f"{N1} 값 있음 %": round(pdg.notna().mean() * 100),
             "n_unique": d[c].nunique(), "예시": ", ".join(map(str, d[c].dropna().unique()[:4]))}
        if num.notna().sum() > 0.5 * d[c].notna().sum():
            r.update({f"{N0} 평균": round(num[d.label == 0].mean(), 2), f"{N1} 평균": round(num[d.label == 1].mean(), 2)})
        rows.append(r)
    tab = pd.DataFrame(rows)
    tab.to_csv(os.path.join(a.out, "participants_columns.csv"), index=False)
    pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 40)
    print(tab.to_string(index=False), "\n")
    cfg = suggest_config(d, tab)
    with open(os.path.join(a.out, "clinical_config_suggested.json"), "w") as f:
        json.dump(cfg, f, indent=2, ensure_ascii=False)
    print("임상 변수 분류 제안:", json.dumps(cfg, ensure_ascii=False, indent=1))
    if a.eeg_root:
        f = find_file(a.eeg_root, d.subject.iloc[0])
        print("\n첫 EEG 파일:", f)
        if f:
            import mne
            r0 = mne.io.read_raw_eeglab(f, preload=True, verbose="ERROR") if f.endswith(".set") else mne.io.read_raw(f, preload=True, verbose="ERROR")
            print(f"채널 {len(r0.ch_names)}개 | fs {r0.info['sfreq']} Hz | 길이 {r0.times[-1]:.1f} s")
            print("채널 이름:", r0.ch_names)
            raw, interp = load_raw(f)
            x = raw.get_data() * 1e6
            print("19채널 매핑 결과:", raw.ch_names, "| 보간한 채널:", interp or "없음")
            print("채널별 표준편차(μV):", dict(zip(raw.ch_names, np.round(x.std(1), 1))))


def suggest_config(d, tab):
    """규칙: HC 값이 거의 없고 PD에만 있으면 누출 / 증상·질병 척도 이름이면 증상 근접 / 나머지는 맥락"""
    ctx, sym, leak = [], [], []
    for _, r in tab.iterrows():
        c = r["column"]
        if r["n_unique"] <= 1 or (r[f"{N0} 값 있음 %"] == 0 and r[f"{N1} 값 있음 %"] == 0):
            continue
        if r["n_unique"] > 2 and pd.to_numeric(d[c], errors="coerce").notna().sum() < 0.5 * d[c].notna().sum():
            continue                                          # 범주가 많은 문자열 열은 제외
        if r[f"{N0} 값 있음 %"] < 20 and r[f"{N1} 값 있음 %"] >= 50:
            leak.append(c)
        elif any(k in c.lower() for k in SYMPTOM_KEYS):
            sym.append(c)
        else:
            ctx.append(c)
    return {"context": ctx, "symptom": sym, "leak": leak}


# ----------------------------------------------------------------------------- extract
def run_extract(a):
    d = select_subjects(load_participants(a.participants, a.label_col, a.task), "full")
    if a.limit:
        d = pd.concat([g.head(a.limit) for _, g in d.groupby("label")])
    fdir = a.feat_dir or os.path.join(a.out, "features"); os.makedirs(fdir, exist_ok=True)
    model = build_cbramod(a.cbramod_repo, a.weights)
    print("device:", next(model.parameters()).device)
    log, t0 = [], time.time()
    old = os.path.join(a.out, "extract_log.csv")
    prev = pd.read_csv(old).set_index(["subject", "cond"]) if os.path.exists(old) else None
    for _, r in d.iterrows():
        cond = CONDITIONS[0]
        fp = os.path.join(fdir, f"{r.subject}_{cond}.npz")
        rec = {"subject": r.subject, "label": int(r.label), "cond": cond}
        if os.path.exists(fp):
            z = np.load(fp); rec.update(status="cached", n_seg=len(z["feat"]), n_flag=int(z["flag"].sum()))
            if prev is not None and (r.subject, cond) in prev.index:
                rec["interp"] = prev.loc[(r.subject, cond)].get("interp", "")
            log.append(rec); continue
        path = find_file(a.eeg_root, r.subject)
        if path is None:
            rec.update(status="no_file"); log.append(rec); continue
        try:
            segs, flag, fs, dur, interp, segmax = preprocess(path)
            if len(segs) == 0:
                rec.update(status="too_short"); log.append(rec); continue
            np.savez_compressed(fp, feat=extract_features(model, segs), flag=flag, segmax=segmax)
            rec.update(status="ok", file=path, fs_orig=fs, dur_s=round(dur, 1), n_seg=len(segs),
                       n_flag=int(flag.sum()), interp="+".join(interp),
                       segmax_median=round(float(np.median(segmax)), 1), over200=int((segmax > 200).sum()), over300=int((segmax > 300).sum()))
        except Exception as e:
            rec.update(status=f"error: {e}", file=path)
        log.append(rec)
        print(rec, flush=True)
    log = pd.DataFrame(log)
    log.to_csv(old, index=False)
    print(log.status.str.split(":").str[0].value_counts().to_string())
    print(f"done in {(time.time() - t0) / 60:.1f} min")


# ----------------------------------------------------------------------------- 실험 1: 교차검증
def metrics(y, p, thr=0.5):
    from sklearn.metrics import roc_auc_score, balanced_accuracy_score, accuracy_score, f1_score, confusion_matrix
    yh = (np.asarray(p) >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yh, labels=[0, 1]).ravel()
    return {"AUROC": roc_auc_score(y, p) if len(set(y)) > 1 else np.nan,
            "BalAcc": balanced_accuracy_score(y, yh), "Acc": accuracy_score(y, yh),
            "Sens": tp / (tp + fn) if tp + fn else np.nan, "Spec": tn / (tn + fp) if tn + fp else np.nan,
            "F1": f1_score(y, yh, zero_division=0), "TN": tn, "FP": fp, "FN": fn, "TP": tp}


def load_feature_table(fdir, subjects, with_flag=False, flag_uv=None):
    X, rows = [], []
    for sid in subjects:
        for cond in CONDITIONS:
            fp = os.path.join(fdir, f"{sid}_{cond}.npz")
            if os.path.exists(fp):
                z = np.load(fp); X.append(z["feat"])
                fl = (z["segmax"] > flag_uv) if (flag_uv and "segmax" in z.files) else z["flag"]   # --flag-uv 로 기준 변경
                rows += [(sid, cond, bool(f)) for f in fl]
    meta = pd.DataFrame(rows, columns=["subject", "cond", "flag"])
    return np.concatenate(X), (meta if with_flag else meta[["subject", "cond"]])


def subject_scores(meta, prob):
    m = meta.assign(p=prob)
    return m.groupby(["subject", "cond"]).p.mean().groupby("subject").mean()


def _fit_eeg(X, y):
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    sc = StandardScaler().fit(X)
    return sc, LogisticRegression(C=C_REG, max_iter=5000).fit(sc.transform(X), y)


def cohort_data(a, cohort, X=None, meta=None, with_flag=False):
    d = select_subjects(load_participants(a.participants, a.label_col, a.task), cohort)
    if X is None:
        X, meta = load_feature_table(a.feat_dir or os.path.join(a.out, "features"), d.subject, with_flag, getattr(a, "flag_uv", None))
    attrs = d.attrs
    d = d[d.subject.isin(meta.subject.unique())].reset_index(drop=True); d.attrs = attrs
    keep = meta.subject.isin(set(d.subject)).values
    return d, X[keep], meta[keep].reset_index(drop=True)


def run_cv(a, X=None, meta=None, seed=None, tag=None, quiet=False):
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import average_precision_score
    seed = SEED if seed is None else seed
    d, X, meta = cohort_data(a, a.cohort, X, meta)
    lab = dict(zip(d.subject, d.label)); y_seg = meta.subject.map(lab).values
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    oof = np.full(len(meta), np.nan); fold_rows = []; subj_fold = {}
    for k, (tr, te) in enumerate(skf.split(d.subject, d.label), 1):
        tr_s, te_s = set(d.subject[tr]), set(d.subject[te])
        itr, ite = meta.subject.isin(tr_s).values, meta.subject.isin(te_s).values
        sc, clf = _fit_eeg(X[itr], y_seg[itr])
        oof[ite] = clf.predict_proba(sc.transform(X[ite]))[:, 1]
        s = subject_scores(meta[ite], oof[ite])
        fold_rows.append({"fold": k, "n_train": len(tr_s), "n_test": len(te_s), **metrics(s.index.map(lab).values, s.values)})
        subj_fold.update({sid: k for sid in te_s})
    s = subject_scores(meta, oof); y = s.index.map(lab).values
    res = {"cohort": a.cohort, "seed": seed, "n_subjects": len(s), "n_HC": int((y == 0).sum()), "n_PD": int((y == 1).sum()),
           "n_recordings": int(meta.groupby(["subject", "cond"]).ngroups), "n_segments": len(meta),
           "subject_level": metrics(y, s.values), "segment_level": metrics(y_seg, oof)}
    res["subject_level"]["AUPRC"] = average_precision_score(y, s.values)
    folds = pd.DataFrame(fold_rows)
    od = os.path.join(a.out, tag or f"cv_{a.cohort}"); os.makedirs(od, exist_ok=True)
    folds.to_csv(os.path.join(od, "folds.csv"), index=False)
    pd.DataFrame({"subject": s.index, "label": y, "p_pd": s.values, "fold": s.index.map(subj_fold)}).to_csv(
        os.path.join(od, "subject_oof.csv"), index=False)
    describe(d).to_csv(os.path.join(od, "demographics.csv"), index=False)
    with open(os.path.join(od, "summary.json"), "w") as f:
        json.dump(res, f, indent=2, default=float)
    if not quiet:
        sl = res["subject_level"]
        print(f"[{a.cohort}] subjects={res['n_subjects']} ({N0} {res['n_HC']}, {N1} {res['n_PD']}), segments={res['n_segments']}")
        print(f"  subject OOF: AUROC {sl['AUROC']:.4f} | BalAcc {sl['BalAcc']*100:.2f}% | Acc {sl['Acc']*100:.2f}% | "
              f"Sens {sl['Sens']*100:.2f}% | Spec {sl['Spec']*100:.2f}% | F1 {sl['F1']*100:.2f}% | AUPRC {sl['AUPRC']:.4f}")
        print(f"  confusion: TN={sl['TN']} FP={sl['FP']} FN={sl['FN']} TP={sl['TP']}")
        print(f"  fold AUROC {folds.AUROC.mean():.4f} ± {folds.AUROC.std():.4f} | "
              f"fold BalAcc {folds.BalAcc.mean()*100:.2f} ± {folds.BalAcc.std()*100:.2f}%p")
        print(f"  segment OOF: AUROC {res['segment_level']['AUROC']:.4f} | BalAcc {res['segment_level']['BalAcc']*100:.2f}%")
    return res


def run_seeds(a):
    rows = []
    for s in range(a.n_seeds):
        r = run_cv(a, seed=s, tag=f"seeds_{a.cohort}/s{s}", quiet=True)["subject_level"]
        rows.append({"seed": s, **{k: round(r[k], 4) for k in ["AUROC", "BalAcc", "Sens", "Spec"]}})
    t = pd.DataFrame(rows); t.to_csv(os.path.join(a.out, f"seeds_{a.cohort}.csv"), index=False)
    print(f"[{a.cohort}] 분할 seed {a.n_seeds}개 반복")
    print(t.drop(columns="seed").agg(["mean", "std", "min", "max"]).round(4).to_string())


# ----------------------------------------------------------------------------- 실험 2: 잡음 점검
def run_noise(a):
    from scipy.stats import mannwhitneyu, spearmanr
    from sklearn.metrics import roc_auc_score
    d, X, meta = cohort_data(a, a.cohort, with_flag=True)
    thr = a.flag_uv or FLAG_UV
    base = f"noise_{a.cohort}" + (f"_uv{int(a.flag_uv)}" if a.flag_uv else "")
    od = os.path.join(a.out, base); os.makedirs(od, exist_ok=True)
    print(f"잡음 기준: 5초 구간 최대 진폭 > {thr:g} μV")
    lab = dict(zip(d.subject, d.label))
    # (1) 전체 vs 잡음 구간 제외
    print("##### 전체 구간"); r_all = run_cv(a, X, meta[["subject", "cond"]], tag=f"{base}/all")
    k = ~meta.flag.values
    print(f"\n##### {thr:g} μV 초과 구간 제외"); r_nf = run_cv(a, X[k], meta[k][["subject", "cond"]].reset_index(drop=True),
                                                  tag=f"{base}/noflag")
    t1 = pd.DataFrame([{"setting": n, "n_subj": r["n_subjects"], "n_seg": r["n_segments"],
                        **{m: round(r["subject_level"][m], 4) for m in ["AUROC", "BalAcc", "Sens", "Spec"]}}
                       for n, r in [("all", r_all), ("noflag", r_nf)]])
    # (2) 그룹별 잡음 비율 + 잡음만으로 분류
    ratio = meta.groupby("subject").flag.mean(); y = ratio.index.map(lab).values
    hc, pdg = ratio[y == 0], ratio[y == 1]
    t2 = pd.DataFrame([{f"{N0}_%": round(hc.mean() * 100, 1), f"{N1}_%": round(pdg.mean() * 100, 1),
                        "많은 쪽": N1 if pdg.mean() > hc.mean() else N0,
                        "MannWhitney_p": round(mannwhitneyu(hc, pdg).pvalue, 4),
                        "noise_only_AUROC": round(roc_auc_score(y, ratio.values), 4)}])
    # (3) 같은 그룹 안에서 라벨 1({N1}) 확률과 잡음 비율의 상관
    oof = pd.read_csv(os.path.join(a.out, f"{base}/all/subject_oof.csv")).set_index("subject")
    oof["noise"] = ratio
    t3 = pd.DataFrame([{"group": N0 if l == 0 else N1, "n": len(g),
                        "spearman_r": round(spearmanr(g.p_pd, g.noise).correlation, 3),
                        "p": round(spearmanr(g.p_pd, g.noise).pvalue, 4)} for l, g in oof.groupby("label")])
    for n, t in [("eval", t1), ("noise_by_group", t2), ("pred_vs_noise", t3)]:
        t.to_csv(os.path.join(od, f"{n}.csv"), index=False)
    print("\n========== (1) 전체 vs 잡음 구간 제외"); print(t1.to_string(index=False))
    print(f"\n========== (2) 그룹별 고진폭 구간 비율 ({thr:g} μV 초과, 피험자별 비율의 평균)"); print(t2.to_string(index=False))
    print(f"\n========== (3) 같은 그룹 안에서 라벨 1({N1}) 확률과 잡음 비율의 상관"); print(t3.to_string(index=False))


# ----------------------------------------------------------------------------- 실험 3·4 공통: 임상 변수, EEG 점수
def load_config(a, d):
    p = a.clinical_config or os.path.join(a.out, "clinical_config.json")
    if not os.path.exists(p):
        p = os.path.join(a.out, "clinical_config_suggested.json")
    with open(p) as f:
        cfg = json.load(f)
    for k in ["context", "symptom", "leak"]:
        cfg[k] = [c for c in cfg.get(k, []) if c in d.columns]
        if d.attrs.get("task") == "cog":                      # 라벨을 만든 MoCA는 입력에서 제외
            cfg[k] = [c for c in cfg[k] if c.lower() != "moca"]
    return cfg


def encode_clinical(d, cfg):
    """맥락·증상: 숫자로 변환(2범주 문자열은 0/1). 누출: 값이 있는지(has_X, 0/1). 결측은 NaN으로 두고 학습 fold 중앙값으로 대체"""
    C = pd.DataFrame(index=d.subject)
    names = {"context": [], "symptom": [], "leak": []}
    for grp in ["context", "symptom"]:
        for c in cfg[grp]:
            v = d.set_index("subject")[c]
            num = pd.to_numeric(v, errors="coerce")
            if num.notna().sum() < 0.5 * v.notna().sum():     # 문자열 -> 2범주만 0/1
                cats = sorted(v.dropna().astype(str).unique())
                if len(cats) != 2:
                    print(f"  [제외] {c}: 범주 {len(cats)}개"); continue
                num = v.astype(str).map({cats[0]: 0.0, cats[1]: 1.0}).where(v.notna())
                print(f"  [변환] {c}: {cats[0]}=0, {cats[1]}=1")
            C[c] = num.values; names[grp].append(c)
    for c in cfg["leak"]:
        C[f"has_{c}"] = d.set_index("subject")[c].notna().astype(float).values
        names["leak"].append(f"has_{c}")
    return C, names


def _logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def eeg_scores(a, d, X, meta):
    """outer 5-fold별 EEG 점수 (logit). 평가 fold = 학습 fold 전체로 학습한 EEG 모델 예측(실험 1과 동일),
    학습 fold = 학습 fold 안의 내부 5-fold out-of-fold 예측 (과적합된 점수로 결합 모델을 학습하는 누수 방지)"""
    from sklearn.model_selection import StratifiedKFold
    cache = os.path.join(a.out, f"clinical_{a.cohort}", "eeg_scores.csv")
    if os.path.exists(cache):
        return pd.read_csv(cache)
    os.makedirs(os.path.dirname(cache), exist_ok=True)
    lab = dict(zip(d.subject, d.label)); y_seg = meta.subject.map(lab).values
    rows = []
    outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    for k, (tr, te) in enumerate(outer.split(d.subject, d.label), 1):
        t0 = time.time()
        trs, tes = d.subject.values[tr], d.subject.values[te]
        itr, ite = meta.subject.isin(set(trs)).values, meta.subject.isin(set(tes)).values
        sc, clf = _fit_eeg(X[itr], y_seg[itr])
        s = subject_scores(meta[ite], clf.predict_proba(sc.transform(X[ite]))[:, 1])
        rows += [{"fold": k, "role": "test", "subject": i, "eeg": _logit(v)} for i, v in s.items()]
        inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
        ytr = d.label.values[tr]
        for itr2, ite2 in inner.split(trs, ytr):
            a_s, b_s = set(trs[itr2]), set(trs[ite2])
            ia, ib = meta.subject.isin(a_s).values, meta.subject.isin(b_s).values
            sc2, clf2 = _fit_eeg(X[ia], y_seg[ia])
            s2 = subject_scores(meta[ib], clf2.predict_proba(sc2.transform(X[ib]))[:, 1])
            rows += [{"fold": k, "role": "train", "subject": i, "eeg": _logit(v)} for i, v in s2.items()]
        print(f"  EEG 점수 fold {k}/5 ({time.time() - t0:.0f} s)", flush=True)
    out = pd.DataFrame(rows); out.to_csv(cache, index=False)
    return out


def _fit_fusion(Xtr, ytr):
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    med = np.nanmedian(Xtr, axis=0); med = np.where(np.isnan(med), 0.0, med)
    Xtr = np.where(np.isnan(Xtr), med, Xtr)
    sc = StandardScaler().fit(Xtr)
    clf = LogisticRegression(C=C_FUSION, max_iter=5000).fit(sc.transform(Xtr), ytr)
    def predict(Xte):
        return clf.predict_proba(sc.transform(np.where(np.isnan(Xte), med, Xte)))[:, 1]
    return predict, clf.coef_[0]


def _frame(scores, C, lab, fold, role):
    s = scores[(scores.fold == fold) & (scores.role == role)].set_index("subject")
    f = C.loc[s.index].copy(); f["eeg"] = s.eeg; f["y"] = [lab[i] for i in s.index]
    return f


def _cv_combo(frames, cols, use_eeg):
    """outer fold별로 학습 fold(EEG 점수 = 내부 OOF)에서 학습, 평가 fold 예측. 전체 OOF 예측과 EEG 비중 반환"""
    feats = (["eeg"] if use_eeg else []) + list(cols)
    preds, shares = [], []
    for tr, te in frames:
        f, coef = _fit_fusion(tr[feats].values, tr.y.values)
        preds.append(pd.Series(f(te[feats].values), index=te.index))
        if use_eeg:
            shares.append(abs(coef[0]) / max(np.abs(coef).sum(), 1e-12))
    return pd.concat(preds), (float(np.mean(shares)) if shares else np.nan)


def _auc(y, p):
    from sklearn.metrics import roc_auc_score
    return roc_auc_score(y, p) if len(set(y)) > 1 else np.nan


def _inner_best(tr, combos, use_eeg=True):
    """학습 fold 안에서만 내부 5-fold로 조합 선택 (중첩 선택)"""
    from sklearn.model_selection import StratifiedKFold
    inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    splits = list(inner.split(tr.index, tr.y))
    best, best_auc = None, -1
    for cols in combos:
        feats = (["eeg"] if use_eeg else []) + list(cols)
        p = np.zeros(len(tr))
        for a_i, b_i in splits:
            f, _ = _fit_fusion(tr.iloc[a_i][feats].values, tr.iloc[a_i].y.values)
            p[b_i] = f(tr.iloc[b_i][feats].values)
        auc = _auc(tr.y.values, p)
        if auc > best_auc:
            best, best_auc = cols, auc
    return best


# ----------------------------------------------------------------------------- 실험 3: 임상정보 조합
def run_clinical(a):
    d, X, meta = cohort_data(a, a.cohort)
    cfg = load_config(a, d)
    print("임상 변수 설정:", cfg)
    C, names = encode_clinical(d, cfg)
    lab = dict(zip(d.subject, d.label))
    od = os.path.join(a.out, f"clinical_{a.cohort}"); os.makedirs(od, exist_ok=True)
    scores = eeg_scores(a, d, X, meta)
    frames = [(_frame(scores, C, lab, k, "train"), _frame(scores, C, lab, k, "test")) for k in range(1, 6)]
    y_all = pd.Series(lab)

    def evaluate(cols, use_eeg):
        p, share = _cv_combo(frames, cols, use_eeg)
        y = y_all.loc[p.index].values
        m = metrics(y, p.values)
        return {"AUROC": round(m["AUROC"], 4), "BalAcc": round(m["BalAcc"], 4), "EEG_share": round(share, 3)}

    # 그룹별 값 보유율
    rows = []
    for grp in ["context", "symptom", "leak"]:
        for c in cfg[grp]:
            v = d.set_index("subject")[c]
            rows.append({"group": grp, "variable": c, f"{N0} 값 있음 %": round(v[d.label.values == 0].notna().mean() * 100),
                         f"{N1} 값 있음 %": round(v[d.label.values == 1].notna().mean() * 100)})
    avail = pd.DataFrame(rows); avail.to_csv(os.path.join(od, "variable_availability.csv"), index=False)

    base = evaluate([], True)
    print(f"\nEEG-only (같은 방식으로 재계산): AUROC {base['AUROC']} | BalAcc {base['BalAcc']}")
    pool = names["context"] + names["symptom"]
    if len(pool) > MAX_COMBO_VARS:
        print(f"  변수 {len(pool)}개 중 앞 {MAX_COMBO_VARS}개만 조합에 사용 (MAX_COMBO_VARS)")
        pool = pool[:MAX_COMBO_VARS]
    combos = [c for r in range(1, len(pool) + 1) for c in itertools.combinations(pool, r)]
    print(f"조합 {len(combos)}개 평가 중...", flush=True)
    rows = []
    for cols in combos:
        clin, fus = evaluate(cols, False), evaluate(cols, True)
        rows.append({"variables": "+".join(cols), "n_vars": len(cols),
                     "has_symptom": any(c in names["symptom"] for c in cols),
                     "clinical_AUROC": clin["AUROC"], "fusion_AUROC": fus["AUROC"], "fusion_BalAcc": fus["BalAcc"],
                     "dAUROC_vs_EEG": round(fus["AUROC"] - base["AUROC"], 4), "EEG_share": fus["EEG_share"]})
    allc = pd.DataFrame(rows).sort_values("fusion_AUROC", ascending=False)
    allc.to_csv(os.path.join(od, "all_combinations.csv"), index=False)
    single = allc[allc.n_vars == 1].sort_values("dAUROC_vs_EEG", ascending=False)
    single.to_csv(os.path.join(od, "single_variables.csv"), index=False)

    # 중첩 선택
    nested = []
    pools = [("맥락만", [c for c in combos if not any(v in names["symptom"] for v in c)])]
    if names["symptom"]:
        pools.append(("맥락 + 증상 근접", combos))
    for name, cands in pools:
        if not cands:
            continue
        preds, chosen = [], []
        for tr, te in frames:
            best = _inner_best(tr, cands)
            feats = ["eeg"] + list(best)
            f, _ = _fit_fusion(tr[feats].values, tr.y.values)
            preds.append(pd.Series(f(te[feats].values), index=te.index)); chosen.append("+".join(best))
        p = pd.concat(preds); m = metrics(y_all.loc[p.index].values, p.values)
        nested.append({"pool": name, "AUROC": round(m["AUROC"], 4), "BalAcc": round(m["BalAcc"], 4),
                       "dAUROC_vs_EEG": round(m["AUROC"] - base["AUROC"], 4), "fold별 선택 조합": " · ".join(chosen)})
    nested = pd.DataFrame(nested); nested.to_csv(os.path.join(od, "nested_selection.csv"), index=False)

    # 누출 대조
    leak_rows = []
    for cols in [[c] for c in names["leak"]] + ([names["leak"]] if len(names["leak"]) > 1 else []):
        clin, fus = evaluate(cols, False), evaluate(cols, True)
        leak_rows.append({"setting": "EEG + " + "+".join(cols), "clinical_AUROC": clin["AUROC"],
                          "fusion_AUROC": fus["AUROC"], "fusion_BalAcc": fus["BalAcc"], "EEG_share": fus["EEG_share"]})
    leak = pd.DataFrame(leak_rows); leak.to_csv(os.path.join(od, "leakage_control.csv"), index=False)
    with open(os.path.join(od, "baseline.json"), "w") as f:
        json.dump({"EEG_only": base, "config": cfg, "encoded": names}, f, indent=2, ensure_ascii=False)

    pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 120)
    print("\n========== 변수별 그룹 값 보유율"); print(avail.to_string(index=False))
    print(f"\n========== (1) 조합 상위 10개 (고정 순위)  | 기준 EEG-only AUROC {base['AUROC']}, BalAcc {base['BalAcc']}")
    print(allc.head(10).to_string(index=False))
    print("\n========== (2) 단일 변수 효과"); print(single.to_string(index=False))
    print("\n========== (3) 중첩 선택 (선택 편향 없는 최상 조합 성능)"); print(nested.to_string(index=False))
    print("\n========== (4) 누출 대조"); print(leak.to_string(index=False) if len(leak) else "누출 변수 없음")


# ----------------------------------------------------------------------------- 실험 4: 정보 교체
def run_swap(a):
    d, X, meta = cohort_data(a, a.cohort)
    cfg = load_config(a, d)
    C, names = encode_clinical(d, cfg)
    lab = dict(zip(d.subject, d.label))
    od = os.path.join(a.out, f"clinical_{a.cohort}"); os.makedirs(od, exist_ok=True)
    scores = eeg_scores(a, d, X, meta)
    frames = [(_frame(scores, C, lab, k, "train"), _frame(scores, C, lab, k, "test")) for k in range(1, 6)]
    models = [("맥락 모델", names["context"])]
    if names["symptom"]:
        models.append(("맥락+증상 모델", names["context"] + names["symptom"]))
    if names["leak"]:
        models.append(("누출 모델", names["leak"]))
    models = [(n, c) for n, c in models if c]
    subj = np.array(list(lab)); ylab = np.array([lab[s] for s in subj])
    rng = np.random.default_rng(SEED)

    def donors(kind):
        out = {}
        for s, y in zip(subj, ylab):
            if kind == "random":   pool = subj[subj != s]
            elif kind == "same":   pool = subj[(ylab == y) & (subj != s)]
            else:                  pool = subj[ylab != y]
            out[s] = rng.choice(pool, N_SWAP, replace=True)
        return out
    donor = {k: donors(k) for k in ["random", "same", "opposite"]}

    res_rows, conflict_rows, per_subj = [], [], []
    for mname, cols in models:
        p_eeg, p_own, p_sw = {}, {}, {k: {} for k in donor}
        for tr, te in frames:
            f0, _ = _fit_fusion(tr[["eeg"]].values, tr.y.values)
            f1, _ = _fit_fusion(tr[["eeg"] + cols].values, tr.y.values)
            p0, p1 = f0(te[["eeg"]].values), f1(te[["eeg"] + cols].values)
            for i, s in enumerate(te.index):
                p_eeg[s], p_own[s] = p0[i], p1[i]
                for kind in donor:
                    clin = C.loc[donor[kind][s], cols].values             # 기증자 N_SWAP명의 임상정보
                    xx = np.column_stack([np.full(N_SWAP, te.loc[s, "eeg"]), clin])
                    p_sw[kind][s] = f1(xx)
        ss = list(p_own); y = np.array([lab[s] for s in ss])
        own = np.array([p_own[s] for s in ss]); eeg = np.array([p_eeg[s] for s in ss])
        res_rows.append({"model": mname, "condition": "1. 미제공 (EEG-only)", "AUROC": round(_auc(y, eeg), 4), "판정 변화 %": np.nan})
        res_rows.append({"model": mname, "condition": "2. 본인 정보", "AUROC": round(_auc(y, own), 4), "판정 변화 %": 0.0})
        for kind, label in [("random", "3. 무작위 다른 환자"), ("same", "4. 같은 그룹 다른 환자"), ("opposite", "5. 반대 그룹 환자")]:
            P = np.array([p_sw[kind][s] for s in ss])                     # [피험자, N_SWAP]
            aucs = [_auc(y, P[:, j]) for j in range(N_SWAP)]
            change = ((P >= 0.5) != (own[:, None] >= 0.5)).mean() * 100
            res_rows.append({"model": mname, "condition": label, "AUROC": round(np.mean(aucs), 4), "판정 변화 %": round(change, 1)})
            if kind == "opposite":
                dec = P >= 0.5; chart = (1 - y)[:, None]                  # 붙인 차트의 그룹
                eeg_dec = (eeg >= 0.5)[:, None]
                differs = np.broadcast_to(eeg_dec != chart, dec.shape)     # EEG-only 판정과 차트 방향이 다른 경우
                follow = ((dec == chart) & differs).sum() / max(differs.sum(), 1) * 100
                conflict_rows.append({"model": mname,
                                      f"{N0} 뇌파 + {N1} 차트 -> {N1} 판정 %": round(dec[y == 0].mean() * 100, 1),
                                      f"(기준) EEG-only {N0} -> {N1} 판정 %": round((eeg[y == 0] >= 0.5).mean() * 100, 1),
                                      f"{N1} 뇌파 + {N0} 차트 -> {N0} 판정 %": round((~dec[y == 1]).mean() * 100, 1),
                                      f"(기준) EEG-only {N1} -> {N0} 판정 %": round((eeg[y == 1] < 0.5).mean() * 100, 1),
                                      "차트를 따른 비율 %": round(follow, 1)})
                for i, s in enumerate(ss):
                    per_subj.append({"model": mname, "subject": s, "label": y[i], "p_eeg_only": eeg[i], "p_own": own[i],
                                     "p_opposite_mean": P[i].mean()})
    res = pd.DataFrame(res_rows); conf = pd.DataFrame(conflict_rows)
    res.to_csv(os.path.join(od, "swap_results.csv"), index=False)
    conf.to_csv(os.path.join(od, "swap_conflict.csv"), index=False)
    pd.DataFrame(per_subj).to_csv(os.path.join(od, "swap_per_subject.csv"), index=False)
    pd.set_option("display.width", 250)
    print("모델별 변수:", {n: c for n, c in models})
    print("\n========== 조건별 AUROC와 판정 변화 (판정 변화 = 본인 정보일 때와 PD/HC 판정이 달라진 비율, 기증자 20회 평균)")
    print(res.pivot(index="condition", columns="model", values=["AUROC", "판정 변화 %"]).to_string())
    print("\n========== 조건 5: EEG와 차트가 충돌할 때")
    print("  (차트를 따른 비율 = EEG-only 판정과 차트 방향이 다른 경우 중, 최종 판정이 차트 쪽으로 바뀐 비율)")
    print(conf.to_string(index=False))


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("step", choices=["inspect", "select", "extract", "cv", "seeds", "noise", "clinical", "swap"])
    ap.add_argument("--participants", required=True)
    ap.add_argument("--out", default="out_ad")
    ap.add_argument("--eeg-root"); ap.add_argument("--cbramod-repo"); ap.add_argument("--weights")
    ap.add_argument("--label-col", default=None)
    ap.add_argument("--limit", type=int, default=0)
    ap.add_argument("--cohort", default="full", choices=["full", "matched"])
    ap.add_argument("--n-seeds", type=int, default=10)
    ap.add_argument("--flag-uv", type=float, default=None, help="실험 2 잡음 기준(μV). 기본 100 (추출 때 저장한 flag)")
    ap.add_argument("--clinical-config", default=None)
    ap.add_argument("--task", default="ad_cn", choices=["ad_cn", "ad_ftd", "ftd_cn", "all"], help="ad_cn: AD vs CN / ad_ftd: AD vs FTD / ftd_cn: FTD vs CN / all: 추출용")
    ap.add_argument("--feat-dir", default=None, help="특징 폴더 (기본: OUT/features)")
    a = ap.parse_args()
    if a.step == "select":
        d0 = load_participants(a.participants, a.label_col, a.task); os.makedirs(a.out, exist_ok=True)
        for c in ["full", "matched"]:
            try:
                d = select_subjects(d0, c); d.to_csv(os.path.join(a.out, f"subjects_{c}.csv"), index=False)
                print(f"== {c}\n{describe(d).to_string(index=False)}\n")
            except RuntimeError as e:
                print(f"== {c}: {e}")
    else:
        {"inspect": run_inspect, "extract": run_extract, "cv": run_cv, "seeds": run_seeds, "noise": run_noise,
         "clinical": run_clinical, "swap": run_swap}[a.step](a)

In [ ]:
%%writefile /content/ad_extra.py
"""
AD-EEG 추가 실험 (연구 계획: EEG 근거를 보존하는 선택적 임상정보 활용)
ad_cbramod.py와 같은 폴더에 두고 실행. 실험 3·4와 같은 outer 5-fold(seed 42), 같은 EEG 점수 캐시를 사용.

  relate   : 실험 B. CBraMod 특징으로 임상 변수(UPDRS·MoCA·CTT·나이 등)를 예측 -> EEG-임상정보 관련성
  preserve : 실험 A. EEG 근거 보존 결합 시제품 3가지를 실험 4의 교체 조건으로 평가
             A1 변화 폭 제한   : 최종 logit = EEG logit + clip(임상 보정, ±δ)
             A2 교체 정보 학습 : 학습 때 다른 환자 차트를 붙인 사본(정답은 본인 것)을 ρ배 추가
             A3 일관성 가중치  : 최종 logit = EEG logit + w·(임상 보정),
                                 w = 차트 값이 '이 EEG로 예측한 값'과 가까울수록 1, 멀수록 0 (폭 τ)

사용 예
  python ad_extra.py relate   --participants P --out OUT --cohort full
  python ad_extra.py preserve --participants P --out OUT --cohort matched
"""
import argparse, json, os, sys
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import ad_cbramod as B

DELTAS = [0.25, 0.5, 1.0, 2.0]
RHOS = [0.5, 1.0, 2.0, 4.0]
TAUS = [0.1, 0.25, 0.5, 1.0, 2.0]          # 0.1·0.25 추가 (더 엄격한 일관성)
RIDGE_ALPHAS = np.logspace(0, 5, 11)


def _sig(x): return 1 / (1 + np.exp(-x))


def subject_features(X, meta, subjects):
    """구간 특징 평균 -> 피험자당 3,800차원"""
    m = pd.DataFrame(X, index=meta.subject.values)
    return m.groupby(level=0).mean().loc[subjects].values


def _outer_splits(d):
    from sklearn.model_selection import StratifiedKFold
    return list(StratifiedKFold(n_splits=5, shuffle=True, random_state=B.SEED).split(d.subject, d.label))


def _ridge_fit(F, y):
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import RidgeCV
    sc = StandardScaler().fit(F)
    r = RidgeCV(alphas=RIDGE_ALPHAS).fit(sc.transform(F), y)
    return lambda G: r.predict(sc.transform(G))


def _ridge_oof(F, y, n=5):
    from sklearn.model_selection import KFold
    p = np.zeros(len(y))
    for a, b in KFold(n_splits=n, shuffle=True, random_state=B.SEED).split(F):
        p[b] = _ridge_fit(F[a], y[a])(F[b])
    return p


def _setup(a):
    d, X, meta = B.cohort_data(a, a.cohort)
    cfg = B.load_config(a, d)
    C, names = B.encode_clinical(d, cfg)
    lab = dict(zip(d.subject, d.label))
    F = subject_features(X, meta, d.subject.values)
    return d, X, meta, C, names, lab, F


# ----------------------------------------------------------------------------- 실험 B: EEG -> 임상 변수
def run_relate(a):
    from scipy.stats import spearmanr
    d, X, meta, C, names, lab, F = _setup(a)
    od = os.path.join(a.out, f"extra_{a.cohort}"); os.makedirs(od, exist_ok=True)
    targets = [c for c in names["context"] + names["symptom"] if C[c].nunique() > 2]
    if a.task == "cog":                                        # 라벨의 연속값(MoCA)도 예측 대상으로 (입력에는 안 씀)
        mc = B._find_col(d, r"^moca$")
        C = C.copy(); C["moca(라벨 원천)"] = pd.to_numeric(d.set_index("subject")[mc], errors="coerce").values
        targets.append("moca(라벨 원천)")
    y_lab = d.label.values
    rows, preds = [], {"subject": d.subject.values, "label": y_lab}
    for v in targets:
        y = C[v].values
        ok = ~np.isnan(y)
        oof = np.full(len(y), np.nan)
        for tr, te in _outer_splits(d):
            tr_ok, te_ok = tr[ok[tr]], te[ok[te]]
            if len(tr_ok) < 10 or len(te_ok) == 0:
                continue
            oof[te_ok] = _ridge_fit(F[tr_ok], y[tr_ok])(F[te_ok])
        preds[f"pred_{v}"] = oof
        r = {"variable": v, "group": "context" if v in names["context"] else ("symptom" if v in names["symptom"] else "label"), "n": int(ok.sum())}
        for gname, g in [("전체", ok), (f"{B.N0} 안에서", ok & (y_lab == 0)), (f"{B.N1} 안에서", ok & (y_lab == 1))]:
            if g.sum() >= 8:
                s = spearmanr(oof[g], y[g])
                r[f"r_{gname}"] = round(s.correlation, 3); r[f"p_{gname}"] = round(s.pvalue, 4)
        res = y[ok] - oof[ok]
        r["R2"] = round(1 - (res ** 2).sum() / ((y[ok] - y[ok].mean()) ** 2).sum(), 3)
        rows.append(r)
    t = pd.DataFrame(rows); t.to_csv(os.path.join(od, "eeg_predicts_clinical.csv"), index=False)
    pd.DataFrame(preds).to_csv(os.path.join(od, "eeg_predicted_values.csv"), index=False)
    pd.set_option("display.width", 250)
    print(f"[{a.cohort}] CBraMod 피험자 평균 특징 -> 임상 변수 (Ridge, outer 5-fold OOF 예측과 실제 값의 Spearman 상관)")
    print(f"  '{B.N1} 안에서'·'{B.N0} 안에서' 상관이 그룹 차이를 뺀 EEG-임상정보 관련성")
    print(t.to_string(index=False))


# ----------------------------------------------------------------------------- 실험 A: EEG 근거 보존 결합
def run_preserve(a):
    d, X, meta, C, names, lab, F = _setup(a)
    od = os.path.join(a.out, f"extra_{a.cohort}"); os.makedirs(od, exist_ok=True)
    scores = B.eeg_scores(a, d, X, meta)
    frames = [(B._frame(scores, C, lab, k, "train"), B._frame(scores, C, lab, k, "test")) for k in range(1, 6)]
    splits = _outer_splits(d)
    pos = {s: i for i, s in enumerate(d.subject.values)}
    models = [(n, c) for n, c in [("맥락+증상 모델", names["context"] + names["symptom"]), ("누출 모델", names["leak"])] if c]
    subj = d.subject.values; ylab = d.label.values
    rng = np.random.default_rng(B.SEED)
    donor = {}
    for kind in ["random", "same", "opposite"]:                 # 실험 4와 같은 방식의 기증자
        donor[kind] = {}
        for s, y in zip(subj, ylab):
            pool = subj[subj != s] if kind == "random" else (subj[(ylab == y) & (subj != s)] if kind == "same" else subj[ylab != y])
            donor[kind][s] = rng.choice(pool, B.N_SWAP, replace=True)

    methods = [("단순 결합", None)] + [("A1 변화 폭 제한", dl) for dl in DELTAS] + \
              [("A2 교체 정보 학습", r) for r in RHOS] + [("A3 일관성 가중치", t) for t in TAUS] + \
              [("A2+A3 조합", (r, t)) for r in [2.0, 4.0] for t in [0.5, 1.0]]
    rows = []
    for mname, cols in models:
        # P[method][kind][subject] = 예측 확률 배열 (own은 길이 1, 교체는 N_SWAP)
        P = {m: {k: {} for k in ["eeg", "own", "random", "same", "opposite"]} for m in methods}
        for k, ((tr, te), (itr, ite)) in enumerate(zip(frames, splits), 1):
            f0, _ = B._fit_fusion(tr[["eeg"]].values, tr.y.values)
            f1, _ = B._fit_fusion(tr[["eeg"] + cols].values, tr.y.values)
            aug = {}
            rng2 = np.random.default_rng(B.SEED + k)
            for r in RHOS:                                         # A2: 다른 환자 차트를 붙인 사본, 정답은 본인 것
                n_copy = int(np.ceil(r)); reps = []
                for c in range(n_copy):
                    take = np.ones(len(tr), bool) if c < int(r) else rng2.random(len(tr)) < (r - int(r))
                    t2 = tr[take].copy()
                    dn = [rng2.choice(tr.index[tr.index != s]) for s in t2.index]
                    t2[cols] = tr.loc[dn, cols].values
                    reps.append(t2)
                big = pd.concat([tr] + reps)
                aug[r], _ = B._fit_fusion(big[["eeg"] + cols].values, big.y.values)
            # A3: 학습 fold에서 EEG -> 각 임상 변수 예측 모델 + 잔차 표준편차(내부 OOF)
            Ftr = F[itr]; gate = []
            for c in cols:
                y = C.loc[subj[itr], c].values.astype(float); ok = ~np.isnan(y)
                med = np.nanmedian(y) if ok.any() else 0.0
                if ok.sum() < 10 or np.nanstd(y) == 0:
                    gate.append(None); continue
                fit = _ridge_fit(Ftr[ok], y[ok])
                sd = np.std(y[ok] - _ridge_oof(Ftr[ok], y[ok])) + 1e-9
                gate.append((fit, sd, med))
            for s in te.index:
                e = te.loc[s, "eeg"]
                own = te.loc[s, cols].values.astype(float)[None, :]
                fself = F[pos[s]][None, :]
                xhat = [g[0](fself)[0] if g else None for g in gate]
                charts = {"own": own}
                for kind in ["random", "same", "opposite"]:
                    charts[kind] = C.loc[donor[kind][s], cols].values.astype(float)
                l_e = np.log(np.clip(f0(np.array([[e]])), 1e-9, 1 - 1e-9) / np.clip(1 - f0(np.array([[e]])), 1e-9, 1))[0]
                for m in methods:
                    P[m]["eeg"][s] = _sig(np.array([l_e]))
                for kind, Z in charts.items():
                    xx = np.column_stack([np.full(len(Z), e), Z])
                    pf = np.clip(f1(xx), 1e-9, 1 - 1e-9); l_f = np.log(pf / (1 - pf))
                    P[("단순 결합", None)][kind][s] = pf
                    for dl in DELTAS:
                        P[("A1 변화 폭 제한", dl)][kind][s] = _sig(l_e + np.clip(l_f - l_e, -dl, dl))
                    for r in RHOS:
                        P[("A2 교체 정보 학습", r)][kind][s] = aug[r](xx)
                    # 일관성: 차트 값이 이 EEG로 예측한 값에서 몇 표준편차 떨어져 있는가 (결측은 중앙값 대체)
                    zs = []
                    for j, g in enumerate(gate):
                        if g is None:
                            continue
                        v = np.where(np.isnan(Z[:, j]), g[2], Z[:, j])
                        zs.append((v - xhat[j]) / g[1])
                    zs = np.array(zs) if zs else np.zeros((1, len(Z)))
                    for t in TAUS:
                        w = np.exp(-(zs ** 2) / (2 * t ** 2)).mean(axis=0)
                        P[("A3 일관성 가중치", t)][kind][s] = _sig(l_e + w * (l_f - l_e))
                    for r in [2.0, 4.0]:                         # A2로 학습한 결합 모델 + A3 일관성 가중치
                        pa = np.clip(aug[r](xx), 1e-9, 1 - 1e-9); l_a = np.log(pa / (1 - pa))
                        for t in [0.5, 1.0]:
                            w = np.exp(-(zs ** 2) / (2 * t ** 2)).mean(axis=0)
                            P[("A2+A3 조합", (r, t))][kind][s] = _sig(l_e + w * (l_a - l_e))
        ss = list(P[methods[0]]["own"]); y = np.array([lab[s] for s in ss])
        eeg = np.array([P[methods[0]]["eeg"][s][0] for s in ss])
        auc_eeg = B._auc(y, eeg)
        rows.append({"model": mname, "method": "EEG-only", "param": "", "AUROC_본인": round(auc_eeg, 4),
                     "AUROC_무작위": round(auc_eeg, 4), "AUROC_같은그룹": round(auc_eeg, 4), "AUROC_반대그룹": round(auc_eeg, 4),
                     "반대: 판정 변화 %": 0.0, "반대: 차트를 따른 %": 0.0, f"{B.N1}뇌파+{B.N0}차트 -> {B.N0} %": round((eeg[y == 1] < 0.5).mean() * 100, 1)})
        for m in methods:
            own = np.array([P[m]["own"][s][0] for s in ss])
            r = {"model": mname, "method": m[0], "param": "" if m[1] is None else (f"ρ{m[1][0]:g}·τ{m[1][1]:g}" if isinstance(m[1], tuple) else m[1]), "AUROC_본인": round(B._auc(y, own), 4)}
            for kind, lbl in [("random", "무작위"), ("same", "같은그룹"), ("opposite", "반대그룹")]:
                M = np.array([P[m][kind][s] for s in ss])
                r[f"AUROC_{lbl}"] = round(np.mean([B._auc(y, M[:, j]) for j in range(M.shape[1])]), 4)
                if kind == "opposite":
                    dec = M >= 0.5; chart = (1 - y)[:, None]
                    differs = np.broadcast_to(((eeg >= 0.5)[:, None]) != chart, dec.shape)
                    r["반대: 판정 변화 %"] = round(((M >= 0.5) != (own[:, None] >= 0.5)).mean() * 100, 1)
                    r["반대: 차트를 따른 %"] = round(((dec == chart) & differs).sum() / max(differs.sum(), 1) * 100, 1)
                    r[f"{B.N1}뇌파+{B.N0}차트 -> {B.N0} %"] = round((~dec[y == 1]).mean() * 100, 1)
            rows.append(r)
    t = pd.DataFrame(rows); t.to_csv(os.path.join(od, "preserve_results.csv"), index=False)
    pd.set_option("display.width", 300); pd.set_option("display.max_rows", 200)
    print(f"[{a.cohort}] EEG 근거 보존 결합 시제품 — 교체 조건별 평가 (기증자 {B.N_SWAP}회 평균)")
    print("  좋은 모델: AUROC_본인은 EEG-only보다 높고, AUROC_반대그룹·무작위는 EEG-only 근처를 유지하며, 차트를 따른 % 가 낮음")
    for mname, g in t.groupby("model", sort=False):
        print(f"\n===== {mname}"); print(g.drop(columns="model").to_string(index=False))


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("step", choices=["relate", "preserve"])
    ap.add_argument("--participants", required=True)
    ap.add_argument("--out", default="out_ad")
    ap.add_argument("--label-col", default=None)
    ap.add_argument("--cohort", default="full", choices=["full", "matched"])
    ap.add_argument("--clinical-config", default=None)
    ap.add_argument("--task", default="ad_cn", choices=["ad_cn", "ad_ftd", "ftd_cn"])
    ap.add_argument("--feat-dir", default=None)
    a = ap.parse_args()
    {"relate": run_relate, "preserve": run_preserve}[a.step](a)

In [ ]:
%%writefile /content/ad_special.py
"""
AD-EEG 고유 실험 (ad_cbramod.py와 같은 폴더에서 실행)

  length   : 기록 길이 학습 곡선. 각 기록의 앞 1·2·4·8분 / 전체만 써서 실험 1 반복 (분할 seed 여러 개)
             -> EEG가 얼마나 있어야 하는가, TDBRAIN(2분)·PD-EEG(4분)·Mumtaz(5분)와 같은 길이로 맞춘 비교
  spectral : 고전적 뇌파 지표 계산 (88명 전원 한 번). 같은 필터 후 Welch PSD
             채널별 상대 밴드 파워(delta 1–4, theta 4–8, alpha 8–13, beta 13–30 Hz) 76개,
             전체 평균 서파화 지표 (delta+theta)/(alpha+beta), theta/alpha 비율, 후두부 알파 피크 주파수(PAF)
  compare  : 과제별로 CBraMod와 고전 지표 비교
             (1) 지표 하나만으로 낸 AUROC  (2) 밴드 파워 76개 + LR (같은 분할 seed)  (3) CBraMod 점수와 서파화 지표의 상관

사용 예
  python ad_special.py length   --participants P --out OUT_TASK --task ad_cn --feat-dir FEAT --cohort matched
  python ad_special.py spectral --participants P --out OUT_ALL  --task all   --eeg-root EEG_ROOT --spec-csv SPEC
  python ad_special.py compare  --participants P --out OUT_TASK --task ad_cn --spec-csv SPEC --cohort matched
"""
import argparse, os, sys
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import ad_cbramod as B

MINUTES = [1, 2, 4, 8, 0]                        # 0 = 전체
BANDS = {"delta": (1, 4), "theta": (4, 8), "alpha": (8, 13), "beta": (13, 30)}
POSTERIOR = ["O1", "O2", "P3", "P4", "Pz"]


# ----------------------------------------------------------------------------- 기록 길이 학습 곡선
def run_length(a):
    d = B.select_subjects(B.load_participants(a.participants, a.label_col, a.task), a.cohort)
    X, meta = B.load_feature_table(a.feat_dir or os.path.join(a.out, "features"), d.subject)
    order = meta.groupby(["subject", "cond"]).cumcount().values     # 기록 안에서의 구간 순서 (시간 순)
    seg_per_min = 60 // B.SEG_SEC
    rows = []
    for m in MINUTES:
        k = order < m * seg_per_min if m else np.ones(len(meta), bool)
        Xm, mm = X[k], meta[k].reset_index(drop=True)
        used = mm.groupby("subject").size() * B.SEG_SEC / 60
        for s in range(a.n_seeds):
            r = B.run_cv(a, Xm, mm, seed=s, tag=f"length_{a.cohort}/m{m}/s{s}", quiet=True)["subject_level"]
            rows.append({"minutes": m or "전체", "seed": s, "AUROC": r["AUROC"], "BalAcc": r["BalAcc"],
                         "평균 사용 길이(분)": used.mean()})
        print(f"  {m or '전체'}분 완료", flush=True)
    t = pd.DataFrame(rows)
    t.to_csv(os.path.join(a.out, f"length_{a.cohort}.csv"), index=False)
    s = t.groupby("minutes", sort=False).agg(AUROC_mean=("AUROC", "mean"), AUROC_sd=("AUROC", "std"),
                                             BalAcc_mean=("BalAcc", "mean"), 사용_길이_분=("평균 사용 길이(분)", "first")).round(4)
    print(f"[{a.task} / {a.cohort}] 기록 길이 학습 곡선 (분할 seed {a.n_seeds}개)")
    print("  참고: TDBRAIN 2분 · PD-EEG 4분 · Mumtaz 5분")
    print(s.to_string())


# ----------------------------------------------------------------------------- 고전 지표 계산
def run_spectral(a):
    import mne
    mne.set_log_level("ERROR")
    d = B.load_participants(a.participants, a.label_col, "all")
    rows = []
    for i, r in d.iterrows():
        f = B.find_file(a.eeg_root, r.subject)
        if f is None:
            print("no file:", r.subject); continue
        raw, _ = B.load_raw(f)
        raw.filter(B.L_FREQ, B.H_FREQ, verbose="ERROR").notch_filter(B.NOTCH, verbose="ERROR")
        sf = raw.info["sfreq"]
        psd = raw.compute_psd(method="welch", fmin=1, fmax=45, n_fft=int(4 * sf), verbose="ERROR")
        P, fr = psd.get_data(), psd.freqs                      # [19, F]
        tot = P[:, (fr >= 1) & (fr < 30)].sum(1)
        row = {"subject": r.subject}
        rel = {}
        for b, (lo, hi) in BANDS.items():
            bp = P[:, (fr >= lo) & (fr < hi)].sum(1) / tot
            rel[b] = bp
            for ch, v in zip(raw.ch_names, bp):
                row[f"{b}_{ch}"] = v
        g = {b: v.mean() for b, v in rel.items()}
        row["slowing"] = (g["delta"] + g["theta"]) / (g["alpha"] + g["beta"])
        row["theta_alpha"] = g["theta"] / g["alpha"]
        post = [raw.ch_names.index(c) for c in POSTERIOR if c in raw.ch_names]
        m = (fr >= 6) & (fr <= 13)
        row["PAF"] = fr[m][P[post][:, m].mean(0).argmax()]
        rows.append(row)
        if i % 10 == 0:
            print(f"  {i + 1}/{len(d)}", flush=True)
    t = pd.DataFrame(rows); os.makedirs(os.path.dirname(a.spec_csv) or ".", exist_ok=True)
    t.to_csv(a.spec_csv, index=False)
    print("저장:", a.spec_csv, t.shape)


# ----------------------------------------------------------------------------- CBraMod vs 고전 지표
def run_compare(a):
    from sklearn.model_selection import StratifiedKFold
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score
    from scipy.stats import spearmanr
    d = B.select_subjects(B.load_participants(a.participants, a.label_col, a.task), a.cohort)
    sp = pd.read_csv(a.spec_csv).set_index("subject")
    d = d[d.subject.isin(sp.index)].reset_index(drop=True)
    y = d.label.values; S = sp.loc[d.subject]
    # (1) 지표 하나
    t1 = pd.DataFrame([{"지표": k, "설명": desc, f"{B.N0} 평균": round(S[k][y == 0].mean(), 3), f"{B.N1} 평균": round(S[k][y == 1].mean(), 3),
                        "AUROC (값이 클수록 라벨 1)": round(roc_auc_score(y, S[k]), 4)}
                       for k, desc in [("slowing", "(delta+theta)/(alpha+beta)"), ("theta_alpha", "theta/alpha"), ("PAF", "후두부 알파 피크 주파수 (Hz)")]])
    # (2) 밴드 파워 76개 + LR, 같은 분할 seed
    cols = [c for c in sp.columns if c.split("_")[0] in BANDS]
    F = S[cols].values; rows = []
    for s in range(a.n_seeds):
        p = np.zeros(len(y))
        for tr, te in StratifiedKFold(n_splits=5, shuffle=True, random_state=s).split(F, y):
            sc = StandardScaler().fit(F[tr])
            p[te] = LogisticRegression(C=1.0, max_iter=5000).fit(sc.transform(F[tr]), y[tr]).predict_proba(sc.transform(F[te]))[:, 1]
        rows.append({"seed": s, "AUROC": roc_auc_score(y, p), "BalAcc": B.metrics(y, p)["BalAcc"]})
    bp = pd.DataFrame(rows)
    t2 = [{"모델": "밴드 파워 76개 + LR (C=1.0)", "AUROC": f"{bp.AUROC.mean():.4f} ± {bp.AUROC.std():.4f}",
           "BalAcc": f"{bp.BalAcc.mean():.4f}"}]
    q = os.path.join(a.out, f"seeds_{a.cohort}.csv")
    if os.path.exists(q):
        cb = pd.read_csv(q).head(a.n_seeds)
        t2.insert(0, {"모델": "frozen CBraMod + LR (실험 1)", "AUROC": f"{cb.AUROC.mean():.4f} ± {cb.AUROC.std():.4f}", "BalAcc": f"{cb.BalAcc.mean():.4f}"})
    t2 = pd.DataFrame(t2)
    # (3) CBraMod 점수와 서파화 지표의 상관
    t3 = None
    q = os.path.join(a.out, f"cv_{a.cohort}", "subject_oof.csv")
    if os.path.exists(q):
        o = pd.read_csv(q).set_index("subject").join(S[["slowing", "theta_alpha", "PAF"]], how="inner")
        rr = []
        for gname, g in [("전체", o), (f"{B.N0} 안에서", o[o.label == 0]), (f"{B.N1} 안에서", o[o.label == 1])]:
            for k in ["slowing", "theta_alpha", "PAF"]:
                r = spearmanr(g.p_pd, g[k]); rr.append({"group": gname, "지표": k, "n": len(g), "spearman_r": round(r.correlation, 3), "p": round(r.pvalue, 4)})
        t3 = pd.DataFrame(rr)
    od = os.path.join(a.out, f"spectral_{a.cohort}"); os.makedirs(od, exist_ok=True)
    t1.to_csv(os.path.join(od, "single_index.csv"), index=False); t2.to_csv(os.path.join(od, "model_compare.csv"), index=False)
    if t3 is not None: t3.to_csv(os.path.join(od, "cbramod_vs_slowing.csv"), index=False)
    pd.set_option("display.width", 250)
    print(f"[{a.task} / {a.cohort}] n = {len(y)} ({B.N0} {(y == 0).sum()}, {B.N1} {(y == 1).sum()})")
    print("\n========== (1) 고전 지표 하나만으로 낸 AUROC"); print(t1.to_string(index=False))
    print(f"\n========== (2) CBraMod vs 밴드 파워 (분할 seed {a.n_seeds}개)"); print(t2.to_string(index=False))
    if t3 is not None:
        print(f"\n========== (3) CBraMod 라벨 1({B.N1}) 확률과 고전 지표의 상관"); print(t3.to_string(index=False))


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("step", choices=["length", "spectral", "compare"])
    ap.add_argument("--participants", required=True)
    ap.add_argument("--out", default="out_ad")
    ap.add_argument("--label-col", default=None)
    ap.add_argument("--task", default="ad_cn", choices=["ad_cn", "ad_ftd", "ftd_cn", "all"])
    ap.add_argument("--cohort", default="full", choices=["full", "matched"])
    ap.add_argument("--feat-dir", default=None)
    ap.add_argument("--eeg-root", default=None)
    ap.add_argument("--spec-csv", default="spectral.csv")
    ap.add_argument("--n-seeds", type=int, default=5)
    a = ap.parse_args()
    {"length": run_length, "spectral": run_spectral, "compare": run_compare}[a.step](a)

In [ ]:
%%writefile /content/ad_more.py
"""
AD-EEG 후속 실험: CBraMod + 고전 뇌파 지표(PAF, theta/alpha, 서파화) 결합
  (1) 실험 3과 같은 쌓기: outer 5-fold(seed 42), 학습 fold의 CBraMod 점수 = 내부 OOF
  (2) 분할 seed 10개: 실험 1 seeds의 seed별 OOF와 같은 분할로 학습한 지표 LR의 OOF를 순위 평균
"""
import argparse, os, sys
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import ad_cbramod as B

INDEX_SETS = [("PAF", ["PAF"]), ("theta/alpha", ["theta_alpha"]), ("PAF + theta/alpha + 서파화", ["PAF", "theta_alpha", "slowing"])]


def run_eegfuse(a):
    from sklearn.model_selection import StratifiedKFold
    from scipy.stats import rankdata
    sp = pd.read_csv(a.spec_csv).set_index("subject")
    d, X, meta = B.cohort_data(a, a.cohort)
    keep = d.subject.isin(sp.index)
    if not keep.all():
        print("고전 지표가 없는 피험자 제외:", list(d.subject[~keep]))
    lab = dict(zip(d.subject, d.label))
    C = sp.loc[d.subject, ["PAF", "theta_alpha", "slowing"]].copy()

    # (1) 실험 3과 같은 쌓기
    scores = B.eeg_scores(a, d, X, meta)
    scores = scores[scores.subject.isin(sp.index)]
    frames = [(B._frame(scores, C, lab, k, "train"), B._frame(scores, C, lab, k, "test")) for k in range(1, 6)]
    p0, _ = B._cv_combo(frames, [], True)
    y0 = np.array([lab[s] for s in p0.index]); auc_eeg = B._auc(y0, p0.values)
    rows = [{"입력": "CBraMod 점수만", "지표만 AUROC": np.nan, "결합 AUROC": round(auc_eeg, 4), "dAUROC_vs_CBraMod": 0.0, "EEG_share": 1.0}]
    for name, cols in INDEX_SETS:
        pi, _ = B._cv_combo(frames, cols, False)
        pf, share = B._cv_combo(frames, cols, True)
        yi = np.array([lab[s] for s in pi.index]); yf = np.array([lab[s] for s in pf.index])
        af = B._auc(yf, pf.values)
        rows.append({"입력": f"CBraMod + {name}", "지표만 AUROC": round(B._auc(yi, pi.values), 4), "결합 AUROC": round(af, 4),
                     "dAUROC_vs_CBraMod": round(af - auc_eeg, 4), "EEG_share": round(share, 3)})
    t1 = pd.DataFrame(rows)

    # (2) 분할 seed 10개: 순위 평균
    rows2 = []
    for s in range(a.n_seeds):
        q = os.path.join(a.out, f"seeds_{a.cohort}", f"s{s}", "subject_oof.csv")
        if not os.path.exists(q):
            print("seed OOF 없음 (실험 1 seeds 단계를 먼저 실행):", q); break
        o = pd.read_csv(q).set_index("subject").loc[d.subject]
        y = o.label.values; cb = o.p_pd.values
        r = {"seed": s, "CBraMod": B._auc(y, cb)}
        for name, cols in INDEX_SETS:
            F = C[cols].values; pi = np.zeros(len(y))
            for tr, te in StratifiedKFold(n_splits=5, shuffle=True, random_state=s).split(d.subject, d.label):
                f, _ = B._fit_fusion(F[tr], y[tr]); pi[te] = f(F[te])
            r[name] = B._auc(y, pi)
            r[f"순위평균: CBraMod + {name}"] = B._auc(y, (rankdata(cb) + rankdata(pi)) / 2)
        rows2.append(r)
    t2 = None
    if rows2:
        s2 = pd.DataFrame(rows2); t2 = []
        for c in [c for c in s2.columns if c != "seed"]:
            row = {"모델": c, "AUROC": f"{s2[c].mean():.4f} ± {s2[c].std():.4f}"}
            if c.startswith("순위평균"):
                dd = s2[c] - s2["CBraMod"]
                row.update({"CBraMod 대비": f"{dd.mean():+.4f}", "CBraMod보다 높은 seed": f"{(dd > 0).sum()}/{len(dd)}"})
            t2.append(row)
        t2 = pd.DataFrame(t2).fillna("")
        s2.to_csv(os.path.join(a.out, f"eegfuse_{a.cohort}_seeds.csv"), index=False)

    od = os.path.join(a.out, f"eegfuse_{a.cohort}"); os.makedirs(od, exist_ok=True)
    t1.to_csv(os.path.join(od, "stacking_seed42.csv"), index=False)
    if t2 is not None: t2.to_csv(os.path.join(od, "rank_average_seeds.csv"), index=False)
    pd.set_option("display.width", 250)
    print(f"[{a.task} / {a.cohort}] n = {len(d)} ({B.N0} {(d.label == 0).sum()}, {B.N1} {(d.label == 1).sum()})")
    print("\n========== (1) 쌓기 결합 (실험 3과 같은 방식, seed 42)"); print(t1.to_string(index=False))
    if t2 is not None:
        print(f"\n========== (2) 분할 seed {len(rows2)}개: 순위 평균 결합"); print(t2.to_string(index=False))


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("step", choices=["eegfuse"])
    ap.add_argument("--participants", required=True)
    ap.add_argument("--out", default="out_ad")
    ap.add_argument("--label-col", default=None)
    ap.add_argument("--task", default="ad_cn", choices=["ad_cn", "ad_ftd", "ftd_cn"])
    ap.add_argument("--cohort", default="full", choices=["full", "matched"])
    ap.add_argument("--feat-dir", default=None)
    ap.add_argument("--spec-csv", required=True)
    ap.add_argument("--n-seeds", type=int, default=10)
    a = ap.parse_args()
    {"eegfuse": run_eegfuse}[a.step](a)

## 4. 데이터 구조 확인 (inspect)
**여기서 한 번 멈추고 출력을 확인하세요.**
- 그룹 수가 CN 29 / AD 36으로 나오는지
- 19채널이 모두 매핑되고 `보간한 채널: 없음`인지
- 샘플링 500 Hz, 길이 수백 초(평균 약 13분)인지

In [ ]:
run("inspect", "ad_cn", f'--eeg-root "{EEG_ROOT}"')

## 4-1. 필터 후 진폭 확인
필터 후 표준편차 중앙값이 대략 5~50 μV면 정상

In [ ]:
import sys, numpy as np, mne
mne.set_log_level('ERROR')
sys.path.insert(0, '/content'); import importlib, ad_cbramod as P; importlib.reload(P)
for sid in ['sub-001', 'sub-002', 'sub-037', 'sub-038', 'sub-066']:        # AD, AD, CN, CN, FTD
    f = P.find_file(EEG_ROOT, sid)
    raw, interp = P.load_raw(f)
    x0 = raw.get_data() * 1e6
    raw.filter(0.3, 75).notch_filter(50)
    x = raw.get_data() * 1e6
    w = int(raw.info['sfreq'] * 5); n = x.shape[1] // w
    seg_max = np.abs(x[:, :n * w].reshape(19, n, w)).max(axis=(0, 2))
    print(f'{sid} | fs {raw.info["sfreq"]} | 길이 {raw.times[-1]:.0f} s | 보간 {interp or "없음"} | '
          f'필터 전 표준편차 중앙값 {np.median(x0.std(1)):.1f} μV -> 필터 후 {np.median(x.std(1)):.1f} μV | '
          f'100 μV 초과 구간 {np.mean(seg_max > 100) * 100:.0f}%')

## 5. 임상 변수 설정 (추가 실험 A·B, 실험 3·4용)
- **context (맥락):** Age, Gender
- **symptom (증상 근접):** MMSE. AD vs CN에서는 CN이 전원 30점이라 사실상 누출, AD vs FTD에서는 정당한 정보

In [ ]:
cfg = {"context": ["Age", "Gender"], "symptom": ["MMSE"], "leak": []}
for t in TASKS:
    json.dump(cfg, open(f'{OUTS[t]}/clinical_config.json', 'w'), indent=2)
print('저장 완료')

## 6. 피험자 선정 확인
- ad_cn: full CN 29 / AD 36, matched 22 / 22
- ad_ftd: full FTD 23 / AD 36, matched 19 / 19

In [ ]:
for t in TASKS:
    print(f'#################### {t}'); run('select', t)

## 7. 특징 추출 — 시험 실행 (그룹당 3명)
88명 전원을 한 번만 추출합니다(`--task all`). `ok`, `fs_orig` 500, `n_seg` 100~250, `interp`가 비어 있으면 다음 셀로

In [ ]:
run("extract", "all", f'--eeg-root "{EEG_ROOT}" --cbramod-repo "{CBRAMOD_REPO}" --weights "{WEIGHTS}" --limit 3')

## 8. 특징 추출 — 전체 (기록이 길어 10~20분 소요, 끊겨도 이어서 진행)

In [ ]:
run("extract", "all", f'--eeg-root "{EEG_ROOT}" --cbramod-repo "{CBRAMOD_REPO}" --weights "{WEIGHTS}"')

In [ ]:
import pandas as pd
pd.set_option('display.width', 250)
log = pd.read_csv(f'{OUTS["all"]}/extract_log.csv')
grp = pd.read_csv(PARTS, sep='\t').set_index('participant_id').Group
log['Group'] = log.subject.map(grp)
print(log.status.str.split(':').str[0].value_counts())
print(log.groupby('Group')[['dur_s', 'n_seg', 'segmax_median', 'n_flag', 'over200', 'over300']].agg(['mean', 'min', 'max']).round(1))
print('보간한 채널:', log['interp'].fillna('없음').replace('', '없음').value_counts().to_dict())
log[~log.status.isin(['ok', 'cached'])]

## 9. 실험 1: EEG-only
주 결과 = matched(나이·성별 1:1), 보조 = full. 분할 seed 10개 반복 포함

In [ ]:
for t in TASKS:
    print(f'\n#################### {t}')
    run('cv', t, '--cohort matched'); run('cv', t, '--cohort full')
    run('seeds', t, '--cohort matched --n-seeds 10'); run('seeds', t, '--cohort full --n-seeds 10')

## 10. 실험 2: 잡음 점검
이 데이터셋은 필터 후 표준편차가 38~70 μV로 커서, **100 μV 기준(다른 데이터셋과 같음)으로는 구간의 84~98%가 잡음으로 표시**됩니다. 그래서 세 가지 기준으로 실행합니다.
- **100 μV:** 다른 데이터셋과 비교용 (거의 모든 구간이 빠짐)
- **200·300 μV:** 이 데이터셋의 진폭 수준에 맞춘 기준 (큰 잡음만 골라냄)

추출할 때 구간별 최대 진폭을 저장해 두므로, 기준을 바꿔도 다시 추출할 필요가 없습니다.

In [ ]:
for t in TASKS:
    for c in ['matched', 'full']:
        for uv in [None, 200, 300]:
            print(f'\n#################### {t} / {c} / 잡음 기준 {uv or 100} μV')
            run('noise', t, f'--cohort {c}' + (f' --flag-uv {uv}' if uv else ''))

## 11. 추가 실험 B: EEG로 임상 변수 예측 (관련성)
그룹 안에서의 상관이 핵심 (예: AD 환자 안에서 EEG가 MMSE를 예측하는가). CN 안의 MMSE는 전원 30점이라 계산되지 않음

In [ ]:
for t in TASKS:
    for c in ['full', 'matched']:
        print(f'\n#################### {t} / {c}'); runx('relate', t, f'--cohort {c}')

## 12. 추가 실험 A: EEG 근거 보존 결합
처음 한 번 EEG 점수를 만드느라 과제·코호트마다 몇 분 걸립니다.
- **AD vs CN:** 결과표의 `단순 결합` 행이 "MMSE 누출 대조 + 정보 교체" 결과입니다 (실험 3·4 대신). EEG가 강할 때 완벽한 지름길이 있어도 EEG를 지킬 수 있는지가 핵심
- **AD vs FTD:** MMSE가 정당하면서 불완전한 정보일 때 이득을 지키면서 강건해지는지

In [ ]:
for t in TASKS:
    for c in ['full', 'matched']:
        print(f'\n#################### {t} / {c}'); runx('preserve', t, f'--cohort {c}')

## 13. 실험 3·4: 임상정보 조합 · 정보 교체
변수 3개(Age, Gender, MMSE) → 7개 조합. 주 분석은 **AD vs FTD**입니다.
AD vs CN은 MMSE가 완벽한 누출이라 결과가 정해져 있지만, 비교용으로 함께 실행합니다.

In [ ]:
for t in ['ad_cn', 'ad_ftd']:
    for c in ['matched', 'full']:
        print(f'\n#################### {t} / {c}')
        run('clinical', t, f'--cohort {c}')
        run('swap', t, f'--cohort {c}')

## 14. AD 고유 1: 기록 길이 학습 곡선
각 기록의 앞 1·2·4·8분 / 전체만 써서 실험 1을 반복 (분할 seed 5개). 이미 뽑은 특징을 앞에서부터 잘라 쓰므로 추가 추출 없음
→ EEG가 얼마나 있어야 하는가, TDBRAIN(2분)·PD-EEG(4분)·Mumtaz(5분)와 같은 길이로 맞춘 비교

In [ ]:
for t in TASKS:
    for c in ['matched', 'full']:
        print(f'\n#################### {t} / {c}'); runs('length', t, f'--cohort {c} --n-seeds 5')

## 15. AD 고유 2: 서파화 지표 비교
AD의 대표적인 뇌파 변화는 서파화(델타·세타 증가, 알파 감소, 알파 피크 주파수 저하)입니다.
- **계산 (88명 한 번, 몇 분):** 같은 필터 후 Welch PSD → 채널별 상대 밴드 파워 76개, 서파화 지표 (delta+theta)/(alpha+beta), theta/alpha, 후두부 알파 피크 주파수(PAF)
- **비교:** (1) 지표 하나만으로 낸 AUROC (2) 밴드 파워 76개 + LR vs CBraMod (같은 분할 seed 10개) (3) CBraMod 점수가 서파화 지표를 따라가는지

In [ ]:
if not os.path.exists(SPEC):
    get_ipython().system(f'python /content/ad_special.py spectral --participants "{PARTS}" --out "{OUTS["all"]}" --task all --eeg-root "{EEG_ROOT}" --spec-csv "{SPEC}"')
for t in TASKS:
    for c in ['matched', 'full']:
        print(f'\n#################### {t} / {c}'); runs('compare', t, f'--cohort {c} --n-seeds 10')

## 16. AD 고유 3: 제공자 전처리본(ICA·ASR 정제) 비교
제공자 전처리본은 0.5–45 Hz 필터, A1·A2 재참조 후 ASR과 ICA로 눈·근육 잡음을 제거한 버전입니다. 같은 프로토콜로 특징을 다시 뽑아 원본 결과와 비교합니다.
**원본과 비슷하면** 성능이 잡음·기록 특성에서 오지 않는다는 강한 근거, **크게 떨어지면** 원본 성능의 일부가 잡음에서 왔다는 뜻입니다.

16-1. 전처리본 받기 (약 2~3 GB)

In [ ]:
DERIV = f'{EEG_ROOT}/derivatives'
if len(glob.glob(f'{DERIV}/sub-*/**/*_eeg.set', recursive=True)) < 88:
    if os.path.isdir(f'{DRIVE_DATA}/derivatives') and len(glob.glob(f'{DRIVE_DATA}/derivatives/sub-*/**/*_eeg.set', recursive=True)) >= 88:
        shutil.copytree(f'{DRIVE_DATA}/derivatives', DERIV, dirs_exist_ok=True)
    else:
        try:
            import openneuro
            openneuro.download(dataset='ds004504', tag='1.0.9', target_dir=EEG_ROOT, include=['derivatives/*'])
        except Exception as e:
            print('openneuro-py 실패 -> AWS S3:', e)
            !pip install -q awscli
            !aws s3 sync --no-sign-request s3://openneuro.org/ds004504/derivatives "$DERIV"
        shutil.copytree(DERIV, f'{DRIVE_DATA}/derivatives', dirs_exist_ok=True)
print('전처리본 .set 파일 수:', len(glob.glob(f'{DERIV}/sub-*/**/*_eeg.set', recursive=True)), '(88개면 정상)')

### 16-2. 전처리본에서 특징 추출 (10~20분)

In [ ]:
run("extract", "all", f'--eeg-root "{DERIV}" --cbramod-repo "{CBRAMOD_REPO}" --weights "{WEIGHTS}"', deriv=True)

### 16-3. 전처리본으로 실험 1·2 반복

In [ ]:
for t in TASKS:
    for c in ['matched', 'full']:
        print(f'\n#################### [전처리본] {t} / {c}')
        run('cv', t, f'--cohort {c}', deriv=True); run('seeds', t, f'--cohort {c} --n-seeds 10', deriv=True); run('noise', t, f'--cohort {c}', deriv=True); run('noise', t, f'--cohort {c} --flag-uv 200', deriv=True)

### 16-4. 원본 vs 전처리본 비교표

In [ ]:
rows = []
for t in TASKS:
    for c in ['matched', 'full']:
        r = {'task': t, 'cohort': c}
        for name, o in [('원본', OUTS[t]), ('전처리본', OUTS_D[t])]:
            q = f'{o}/seeds_{c}.csv'
            if os.path.exists(q):
                s = pd.read_csv(q); r[f'{name} AUROC'] = f'{s.AUROC.mean():.4f} ± {s.AUROC.std():.4f}'; r[f'{name} BalAcc'] = round(s.BalAcc.mean(), 4)
            for uv, sub in [(100, ''), (200, '_uv200')]:
                q = f'{o}/noise_{c}{sub}/noise_by_group.csv'
                if os.path.exists(q):
                    n = pd.read_csv(q).iloc[0]; r[f'{name} >{uv}μV 구간 % (0/1)'] = f'{n.iloc[0]} / {n.iloc[1]}'
        rows.append(r)
print(pd.DataFrame(rows).to_string(index=False))
for name, o in [('원본', OUTS['all']), ('전처리본', OUTS_D['all'])]:
    q = f'{o}/extract_log.csv'
    if os.path.exists(q):
        l = pd.read_csv(q); print(f'{name}: 구간 최대 진폭 중앙값 {l.segmax_median.mean():.0f} μV, 100/200/300 μV 초과 구간 '
                                  f'{(l.n_flag / l.n_seg).mean() * 100:.1f} / {(l.over200 / l.n_seg).mean() * 100:.1f} / {(l.over300 / l.n_seg).mean() * 100:.1f}%, 기록당 구간 {l.n_seg.mean():.0f}개')

### 16-5. 전처리본으로 실험 3·4·A 반복 (AD vs CN)
잡음을 제거한 뒤에도 "단순 결합은 MMSE에 끌려가고, 보존 결합은 EEG 판단을 지킨다"는 결론이 같은지 확인합니다.

In [ ]:
cfg = {"context": ["Age", "Gender"], "symptom": ["MMSE"], "leak": []}
for t in TASKS:
    json.dump(cfg, open(f'{OUTS_D[t]}/clinical_config.json', 'w'), indent=2)

def runx_d(step, task, extra=''):            # ad_extra.py를 전처리본 특징으로 실행
    get_ipython().system(f'python /content/ad_extra.py {step} --participants "{PARTS}" --out "{OUTS_D[task]}" --task {task} --feat-dir "{FEAT_DERIV}" {extra}')

for c in ['matched', 'full']:
    print(f'\n#################### [전처리본] ad_cn / {c}')
    run('clinical', 'ad_cn', f'--cohort {c}', deriv=True)   # 실험 3
    run('swap', 'ad_cn', f'--cohort {c}', deriv=True)       # 실험 4
    runx_d('preserve', 'ad_cn', f'--cohort {c}')            # 실험 A

In [ ]:
pick = [('EEG-only', ''), ('단순 결합', ''), ('A1 변화 폭 제한', '0.25'), ('A1 변화 폭 제한', '0.5'),
        ('A3 일관성 가중치', '0.5'), ('A2+A3 조합', 'ρ2·τ0.5'), ('A2+A3 조합', 'ρ4·τ0.5')]
cols = ['AUROC_본인', 'AUROC_같은그룹', 'AUROC_반대그룹', '반대: 차트를 따른 %']
for c in ['matched', 'full']:
    rows = []
    for name, o in [('원본', OUTS['ad_cn']), ('전처리본', OUTS_D['ad_cn'])]:
        q = f'{o}/extra_{c}/preserve_results.csv'
        if not os.path.exists(q): print('없음:', q); continue
        t = pd.read_csv(q, dtype={'param': str}).fillna({'param': ''})
        t = t[t.model == '맥락+증상 모델']
        for m, p in pick:
            r = t[(t.method == m) & (t.param.astype(str).str.replace('.0', '', regex=False) == p.replace('.0', ''))]
            if len(r): rows.append({'데이터': name, '방법': f'{m} {p}'.strip(), **r.iloc[0][cols].to_dict()})
    print(f'\n========== AD vs CN / {c}: 원본 vs 전처리본 (맥락+증상 모델)')
    print(pd.DataFrame(rows).to_string(index=False))

### 16-6. 전처리본으로 추가 실험 B (EEG–임상정보 관련성)

In [ ]:
for t in TASKS:
    for c in ['full', 'matched']:
        print(f'\n#################### [전처리본] {t} / {c}'); runx_d('relate', t, f'--cohort {c}')

## 17. 후속: CBraMod + 고전 뇌파 지표 결합 (`ad_more.py`)
(1) 실험 3과 같은 쌓기 결합(seed 42) (2) 분할 seed 10개에서 CBraMod 점수와 지표 모델 점수의 순위 평균. 원본·전처리본 모두 실행합니다.

In [ ]:
SPEC_D = f'{WORK}/spectral_deriv.csv'
DERIV = f'{EEG_ROOT}/derivatives'
if len(glob.glob(f'{DERIV}/sub-*/**/*_eeg.set', recursive=True)) < 88:
    shutil.copytree(f'{DRIVE_DATA}/derivatives', DERIV, dirs_exist_ok=True)
if not os.path.exists(SPEC_D):
    get_ipython().system(f'python /content/ad_special.py spectral --participants "{PARTS}" --out "{OUTS_D["all"]}" --task all --eeg-root "{DERIV}" --spec-csv "{SPEC_D}"')

for t in TASKS:
    for c in ['matched', 'full']:
        for name, o, f, s in [('원본', OUTS[t], FEAT, SPEC), ('전처리본', OUTS_D[t], FEAT_DERIV, SPEC_D)]:
            print(f'\n#################### [{name}] {t} / {c}')
            get_ipython().system(f'python /content/ad_more.py eegfuse --participants "{PARTS}" --out "{o}" --task {t} --feat-dir "{f}" --spec-csv "{s}" --cohort {c} --n-seeds 10')

## 18. 보고서용 요약

In [ ]:
pd.set_option('display.max_colwidth', 100)
for t in TASKS:
    for c in ['matched', 'full']:
        p = f'{OUTS[t]}/cv_{c}/summary.json'
        if not os.path.exists(p): continue
        s = json.load(open(p)); sl = s['subject_level']
        print(f'\n#################### {t} / {c}: n={s["n_subjects"]} ({s["n_HC"]} / {s["n_PD"]}), AUROC {sl["AUROC"]:.4f}, BalAcc {sl["BalAcc"]:.4f}')
        for n in [f'seeds_{c}', f'noise_{c}/eval', f'noise_{c}/noise_by_group', f'noise_{c}_uv200/eval', f'noise_{c}_uv200/noise_by_group', f'noise_{c}_uv300/eval', f'extra_{c}/eeg_predicts_clinical', f'extra_{c}/preserve_results',
                  f'clinical_{c}/nested_selection', f'clinical_{c}/swap_results', f'length_{c}', f'spectral_{c}/single_index', f'spectral_{c}/model_compare']:
            q = f'{OUTS[t]}/{n}.csv'
            if os.path.exists(q):
                x = pd.read_csv(q)
                if n.startswith('seeds') or n.startswith('length'): x = x.drop(columns='seed').groupby(x.columns[0], sort=False).mean() if n.startswith('length') else x.drop(columns='seed').agg(['mean', 'std'])
                print(f'[{n}]'); print(x.round(4).to_string())